## Things to do...

1. Add position labelling for players (probs easiest via distance to own goal at starting time of match)
2. Add 2021 data to AWS. Will need to find match schedule for 2021 as current kamper.xlsx is only for 2020 matches
3. Compile statistics from multiple matches (should have 4 matches across 2 years where teams go head to head)
4. Adding length CCDF to Markov model and stochastic models via centroid speed + duration of runs
5. Add variable to model truncation, distance/duration to field limits, opponent pressure, something like that...
6. Team coupling analysis: team polarisations are highly correlated, mess around with some 'modes'.
7. General refactoring
8. Feel free to improve animation at the end

# Notebook overview: collective transport and run survival in football GPS data

This notebook analyses football GPS tracking data using ideas from movement ecology, active matter, and survival analysis. The central idea is to treat a football team as a collective moving object. We study both individual player motion and the motion of the team centroid, then ask how collective order affects the persistence and termination of centroid runs.

The pipeline is: $\text{raw GPS} \rightarrow \text{pitch coordinates} \rightarrow \text{active-player paths} \rightarrow \text{centroid trajectories} \rightarrow \text{run segmentation} \rightarrow \text{collective order} \rightarrow \text{hazard / survival models}$.

The main scientific question is: $\boxed{\text{Do coherent, polarised team states create longer-lived and longer-ranged centroid transport?}}$

In plain language: when players move in a more coordinated direction, does the team centroid travel further and remain in a persistent run for longer?

---

## 1. Raw GPS loading and match selection

The data are stored as player-level parquet files in S3. We first identify which fixture we want to analyse, load the corresponding day of tracking data, and attach readable metadata such as `team`, `source_key`, `player_name`, `timestamp`, `lat`, and `lon`.

For head-to-head matches, we load both tracked teams and concatenate them into a single table. For most exploratory work, the data are downsampled to 1 Hz to reduce memory usage and make the downstream analysis more stable.

The important raw data object is usually `df_raw`. This contains one row per player per timestamp before pitch calibration and match filtering.

---

## 2. Pitch calibration: converting GPS to pitch coordinates

Raw latitude/longitude values are inconvenient for movement analysis. We therefore convert them into pitch-aligned Cartesian coordinates: $(\text{lat}, \text{lon}) \rightarrow (x_m, y_m)$, where $x_m$ and $y_m$ are metres relative to the pitch centre.

The pitch calibration does three things:

1. finds the correct pitch polygon;
2. uses the pitch centre as the spatial origin;
3. rotates the coordinate system so that the long axis of the pitch is aligned with the $x$-axis.

After this step, each GPS point has coordinates $\mathbf{r}_i(t) = (x_i(t), y_i(t))$.

The pitch-centred dataframe is usually `df_xy`. This coordinate system allows us to measure distances, speeds, run lengths, centroid motion, and heading angles in metres.

---

## 3. Match phases and active-player labelling

The raw day-level data contain more than just active match play. Players may appear before kickoff, during half-time, after the match, or while warming up near the pitch.

We therefore label match phases as $\text{pre}$, $1H$, $HT$, $2H$, and $\text{post}$. The main analysis uses only the two halves, $1H \cup 2H$.

We also label each player sample as either $\text{active}$ or $\text{bench}$.

The active-player classifier uses pitch geometry and hysteresis. The idea is:

1. a player becomes active only after being inside the active playing area for long enough;
2. a player becomes bench only after being off the pitch for long enough.

This avoids noisy one-frame switches caused by GPS jitter.

The active-only dataframe is `df_active`. This is the dataframe used for the movement and collective-order analysis.

---

## 4. Building continuous player paths

The tracking data are split into continuous paths. A new path is started when:

1. there is a large time gap;
2. the match phase changes;
3. the player changes;
4. for head-to-head data, the team/source context changes.

For each active player, we build continuous trajectories $\mathbf{r}_i(t) = (x_i(t), y_i(t))$.

The resulting dataframe is `df_paths`.

This is important because movement statistics such as mean squared displacement and run segmentation should never jump across discontinuities, substitutions, half-time gaps, or missing data.

---

## 5. Team centroid

For each team and time point, we compute the team centroid as $\mathbf{C}(t) = \frac{1}{N(t)} \sum_{i=1}^{N(t)} \mathbf{r}_i(t)$, where $N(t)$ is the number of active players contributing at time $t$.

The centroid is the team-level analogue of an individual trajectory. It captures the large-scale translation of the team shape across the pitch.

The centroid time series is usually `centroid_ts`.

The centroid trajectory is then split into continuous centroid paths, just like player trajectories.

---

## 6. Absolute, relative, and centroid-frame motion

A player’s position can be decomposed as $\mathbf{r}_i(t) = \mathbf{C}(t) + \boldsymbol{\rho}_i(t)$, where $\mathbf{C}(t)$ is the team centroid and $\boldsymbol{\rho}_i(t)$ is the player’s position relative to the centroid.

This lets us separate two kinds of motion:

1. **Collective translation**: the whole team moving across the pitch.
2. **Internal rearrangement**: players moving relative to their team shape.

For displacements over lag $\tau$, we have $\Delta \mathbf{r}_i = \Delta \mathbf{C} + \Delta \boldsymbol{\rho}_i$.

Therefore the player mean squared displacement decomposes as $\left\langle |\Delta \mathbf{r}_i|^2 \right\rangle = \left\langle |\Delta \mathbf{C}|^2 \right\rangle + \left\langle |\Delta \boldsymbol{\rho}_i|^2 \right\rangle + 2\left\langle \Delta \mathbf{C}\cdot \Delta \boldsymbol{\rho}_i \right\rangle$.

This decomposition tells us whether player superdiffusion mostly comes from individual wandering or from coherent transport of the whole team centroid.

---

## 7. Mean squared displacement and superdiffusion

For a trajectory $\mathbf{r}(t)$, the mean squared displacement is $\mathrm{MSD}(\tau) = \left\langle |\mathbf{r}(t+\tau)-\mathbf{r}(t)|^2 \right\rangle_t$.

We often fit a power law $\mathrm{MSD}(\tau) \sim \tau^\alpha$.

Interpretation:

* $\alpha = 1$ means Brownian-like diffusion.
* $\alpha > 1$ means superdiffusion or persistent transport.
* $\alpha = 2$ means ballistic motion.

In this notebook, we compare MSDs for:

1. absolute player motion;
2. player motion relative to the centroid;
3. centroid motion.

The working hypothesis is that a substantial part of the observed superdiffusive behaviour comes from the centroid component, i.e. from coordinated team-level transport.

---

## 8. Turning-based run segmentation

We segment trajectories into “runs” using a turning-angle rule.

Given consecutive displacement vectors $\Delta \mathbf{r}*k = \mathbf{r}(t_k)-\mathbf{r}(t*{k-1})$, we compute the turning angle $\theta_k = \cos^{-1}\left(\frac{\Delta \mathbf{r}_{k-1}\cdot \Delta \mathbf{r}*k}{|\Delta \mathbf{r}*{k-1}|,|\Delta \mathbf{r}_k|}\right)$.

A run continues while the turning angle remains below a threshold, $\theta_k \leq \theta_{\mathrm{max}}$.

When $\theta_k > \theta_{\mathrm{max}}$, the current run ends and a new one begins.

For each run we compute duration $T = t_{\mathrm{end}} - t_{\mathrm{start}}$, arc length $L = \sum_k |\Delta \mathbf{r}_k|$, and mean run speed $\bar v = L/T$.

For centroid runs, these quantities describe persistent team-level transport episodes.

---

## 9. Empirical run distributions and heavy tails

For run durations $T$ and run lengths $L$, we plot complementary cumulative distribution functions $P(T \geq t)$ and $P(L \geq \ell)$.

These are useful because broad or heavy-tailed distributions are easier to see on log-log CCDF plots.

The broad empirical tails suggest that centroid transport is not characterised by one typical run scale. Instead, many short runs are mixed with occasional long persistent episodes.

A key goal of the notebook is to explain these long episodes mechanistically rather than only fitting their distributions.

---

## 10. Collective order: polarisation, milling, and speed

At each time point, we compute the velocity direction of every active player as $\hat{\mathbf{v}}_i(t) = \mathbf{v}_i(t)/|\mathbf{v}_i(t)|$.

The team polarisation is $p(t) = \left|\frac{1}{N(t)}\sum_{i=1}^{N(t)} \hat{\mathbf{v}}_i(t)\right|$.

Interpretation:

* $p(t)\approx 1$ means players are moving in the same direction.
* $p(t)\approx 0$ means player directions cancel, producing disordered collective movement.

We also compute a milling or rotational order parameter. Let $\hat{\boldsymbol{\rho}}_i(t) = \frac{\mathbf{r}_i(t)-\mathbf{C}(t)}{|\mathbf{r}_i(t)-\mathbf{C}(t)|}$ be the unit vector from the centroid to player $i$.

Then $m(t) = \left|\frac{1}{N(t)}\sum_i \left(\hat{\boldsymbol{\rho}}_i(t) \times \hat{\mathbf{v}}_i(t)\right)_z\right|$.

Large $m(t)$ means players are moving tangentially around the centroid.

The notebook mainly focuses on polarisation $p(t)$, because it is directly connected to coherent team translation.

The collective-order dataframe is `df_pmv`, with columns such as `p_group`, `m_group`, and `v_group_mps`. Here, `v_group_mps` is the mean individual/player speed within the team at that time point.

---

## 11. Run-level collective order

For each centroid run, we attach summaries of collective order.

Start polarisation is $p_{\mathrm{start}} = p(t_{\mathrm{start}})$.

Early polarisation, for example over the first 3 seconds, is $p_{\mathrm{early}} = \frac{1}{3}\int_{t_{\mathrm{start}}}^{t_{\mathrm{start}}+3} p(t),dt$.

Realised mean polarisation across the whole run is $p_{\mathrm{mean}} = \frac{1}{T}\int_{t_{\mathrm{start}}}^{t_{\mathrm{end}}} p(t),dt$.

The important interpretation is that $p_{\mathrm{mean}}$ is retrospective. It describes the realised order during a run. It should not be interpreted as information known at the start of the run.

Therefore:

1. $p_{\mathrm{mean}}$ is useful for describing the phenotype of long runs.
2. $p_{\mathrm{start}}$, $p_{\mathrm{early}}$, or strictly-past $p$ are better for prospective hazard claims.

We split runs into low/mid/high order states using team-specific terciles of $p_{\mathrm{mean}}$ or current $p(t)$. This avoids comparing raw polarisation values across teams with different baselines.

---

## 12. Figure 2 logic: transport phenotype

Figure 2 is the transport phenotype.

It asks:

1. Do players and centroids have broad run-length and run-duration distributions?
2. Is centroid motion itself superdiffusive?
3. How much of player MSD is explained by centroid motion versus relative motion?

The key conceptual result is $\mathbf{r}_i(t) = \mathbf{C}(t) + \boldsymbol{\rho}_i(t)$.

If the centroid term explains a large fraction of player MSD, then team-level transport is a major component of player movement statistics.

This supports the idea that football movement should not only be analysed as independent individual trajectories. It has a collective active-matter component.

---

## 13. Figure 3 logic: collective order and transport

Figure 3 links collective order to centroid transport.

The main questions are:

1. Are high-polarisation frames faster?
2. Do high-order centroid runs last longer?
3. Do high-order centroid runs travel further?

We compare low/mid/high realised order states and plot $P(T \geq t \mid p_{\mathrm{mean}} \in \text{state})$ and $P(L \geq \ell \mid p_{\mathrm{mean}} \in \text{state})$.

The expected pattern is $\text{high } p_{\mathrm{mean}} \Rightarrow \text{longer duration tails and longer length tails}$.

This does not by itself prove causality, because $p_{\mathrm{mean}}$ is measured over the whole run. But it shows that long centroid transport episodes are realised as coherent collective-order episodes.

We also add speed to separate two effects: $L = \int_0^T v_c(t),dt \approx T\bar v_c$.

So high-order runs may travel further because:

1. they last longer;
2. they move faster;
3. both.

The notebook tests this by comparing run duration $T$, run speed $L/T$, and run length $L$ across order states.

---

## 14. Hazard modelling: from run descriptions to termination risk

The hazard analysis treats each centroid run as a survival process.

For a run of age $a$, the hazard is the instantaneous probability per unit time that the run terminates: $h(a) = \lim_{\Delta t \to 0} \frac{P(a \leq T < a+\Delta t \mid T \geq a)}{\Delta t}$.

In discrete 1-second intervals, the empirical hazard is estimated as $\hat h(a) = \frac{\text{number of run terminations in age bin } a}{\text{total exposure time in age bin } a}$.

The hazard table has one row per run-age interval and is usually called `hazard_intervals`.

Each row contains:

1. run ID;
2. team;
3. match phase;
4. interval start/end;
5. run age;
6. exposure time $\Delta t$;
7. event indicator;
8. current collective order;
9. current speed.

The event indicator is $\text{event}=1$ only for the terminal interval of a run.

---

## 15. Inverse-age hazard model

Empirically, run termination risk often decreases with run age. Short unstable runs die early, while surviving runs are progressively enriched for more stable contexts.

We model this using an inverse-age hazard: $h_0(a) = \lambda_\infty + \frac{\mu}{a_0+a}$.

Parameters:

1. $\lambda_\infty$: long-age hazard floor;
2. $\mu$: strength of early-age decay;
3. $a_0$: small offset, usually fixed to 1 second.

The corresponding survival curve is $S(a) = \exp(-\lambda_\infty a)\left(\frac{a_0}{a_0+a}\right)^\mu$.

Interpretation:

1. early ages have high termination risk;
2. hazard falls as fragile runs are removed;
3. long-lived runs survive in more persistent contexts;
4. $\lambda_\infty$ creates eventual exponential tempering.

This model is useful because it gives broad, heavy-tailed-like survival over intermediate times, but still allows finite truncation.

---

## 16. Hidden-fragility interpretation

One way to interpret the inverse-age hazard is through hidden heterogeneity.

Suppose each run has an unobserved fragility $r$, representing local pressure, poor shape, limited space, or tactical instability. Conditional on $r$, survival is approximately exponential: $S(a\mid r) = \exp[-(\lambda_\infty+r)a]$.

If $r$ varies across runs, high-fragility runs terminate early. The surviving population becomes biased toward low-fragility runs, so the observed hazard decreases with age.

A Gamma frailty assumption gives the closed form $S(a) = \exp(-\lambda_\infty a)\left(\frac{\beta}{\beta+a}\right)^\mu$, and therefore $h(a) = \lambda_\infty + \frac{\mu}{\beta+a}$.

We do not need to claim that the true hidden rates are literally Gamma distributed. The Gamma model is a minimal mathematical closure that gives an interpretable inverse-age hazard.

The important idea is $\boxed{\text{apparent run-age memory can arise from selection over hidden contextual fragilities.}}$

---

## 17. Order-dependent hazard

To test whether collective order affects run termination, we extend the baseline hazard with covariates: $h(a, x) = h_0(a)\exp(\beta^\top x)$.

For example, $h(a,p) = h_0(a)\exp(\beta_p z_p)$, where $z_p = \frac{p-\bar p}{\sigma_p}$ is the z-scored polarisation.

The hazard ratio for a one-standard-deviation increase in polarisation is $HR_p = \exp(\beta_p)$.

Interpretation:

* $HR_p < 1$ means higher polarisation lowers run termination risk.
* $HR_p > 1$ means higher polarisation increases run termination risk.

We also include speed controls: $h(a,p,v) = h_0(a)\exp(\beta_p z_p + \beta_v z_v)$.

This asks whether order matters beyond simply moving fast.

---

## 18. Strictly-past covariates

For a prospective interpretation, we avoid using same-interval or whole-run information. Instead, we use strictly-past rolling averages.

For a window $W$, the past polarisation at age $a$ is $\bar p_{\mathrm{past},W}(a) = \frac{\sum_{j: a-W \leq a_j < a} p_j \Delta t_j}{\sum_{j: a-W \leq a_j < a} \Delta t_j}$.

The current interval is excluded.

This gives a cleaner claim: $\boxed{\text{recent collective order predicts subsequent run survival.}}$

This is stronger than saying that long runs have high mean order.

---

## 19. Figure 4 logic: hazard and state switching

Figure 4 is the mechanistic figure.

It usually contains:

1. empirical hazard as a function of run age;
2. inverse-age hazard fits;
3. hazard split by low/mid/high order states;
4. transition probabilities among order states;
5. simulation of a killed state process;
6. comparison between simulated and empirical duration distributions.

The state process discretises polarisation into $s(t) \in {\text{low}, \text{mid}, \text{high}}$.

We estimate a transition matrix $P_{ij} = P(s_{t+\Delta t}=j \mid s_t=i)$.

We then combine this state process with state-dependent killing: $h(a,s) = \lambda_\infty(s) + \frac{\mu(s)}{a_0+a}$.

The goal is to test whether $\text{state persistence} + \text{state-dependent killing} + \text{inverse-age memory}$ can reconstruct the observed run-duration distribution.

If it works, the message is $\boxed{\text{broad centroid-run tails arise from collective state dynamics, not just static run statistics.}}$

---

## 20. Figure 5 logic: continuous stochastic order model

Figure 5 replaces discrete low/mid/high states with a continuous stochastic process for polarisation.

We model $p(t)$ as a one-dimensional stochastic process: $dp = b(p),dt + \sigma(p),dW_t$, where $b(p)$ is the drift, $\sigma(p)$ is the state-dependent noise amplitude, and $W_t$ is Brownian noise.

Run termination is modelled as stochastic killing with hazard $h(a,p) = h_0(a)\phi(p)$.

Here, $h_0(a) = \lambda_\infty + \frac{\mu}{a_0+a}$, and $\phi(p)$ modulates termination risk according to collective order.

Typically we expect $\phi(p)$ to decrease as $p$ increases, meaning high-order states are protective.

The simulation procedure is:

1. simulate $p(t)$;
2. compute the age- and order-dependent killing hazard;
3. terminate the run stochastically;
4. collect simulated run durations and run-mean polarisation;
5. compare simulated distributions to empirical ones.

The model is successful if it reproduces both $P(T \geq t)$ and the distribution of realised run-order phenotypes.

---

## 21. Velocity: mediator and control

Velocity enters the analysis in two different ways.

First, speed is a mediator of spatial displacement: $L = \int_0^T v_c(t),dt$. Even if two runs have the same duration, the faster one travels further.

Second, speed can be a hazard control. We ask whether polarisation still lowers termination risk after controlling for recent speed: $h(a,p,v) = h_0(a)\exp(\beta_p z_p+\beta_v z_v)$.

There are several speed variables, and they should not be confused.

Group/player speed is $v_{\mathrm{group}}(t) = \frac{1}{N}\sum_i |\mathbf{v}_i(t)|$.

Centroid speed is $v_c(t) = |\dot{\mathbf{C}}(t)|$.

Run-mean centroid speed is $\bar v_{\mathrm{run}} = L/T$.

The key interpretation is $\boxed{\text{order can both increase coherent displacement speed and reduce termination risk.}}$

If polarisation remains protective after speed controls, then order is not merely a proxy for fast movement.

---

## 22. Optional head-to-head coupling analysis

For matches where both teams are tracked, we can analyse competitive coupling between teams.

Let the two team centroids be $\mathbf{C}_A(t)$ and $\mathbf{C}_B(t)$.

We define inter-team distance as $d_{AB}(t) = |\mathbf{C}_A(t)-\mathbf{C}_B(t)|$.

The closing speed is $v_{\mathrm{close}}(t) = -\frac{d}{dt}d_{AB}(t)$.

So $v_{\mathrm{close}} > 0$ means the two centroids are moving closer together.

We also compute centroid velocity alignment as $A_{AB}(t) = \hat{\mathbf{v}}_A(t)\cdot \hat{\mathbf{v}}_B(t)$.

Interpretation:

* $A_{AB}\approx 1$ means teams are moving in the same direction.
* $A_{AB}\approx -1$ means teams are moving in opposite directions.

We also define order imbalance as $\Delta p(t) = p_A(t)-p_B(t)$.

For team $A$, a coupled hazard model can be written as $h_A(a) = h_0(a)\exp\left(\beta_{\mathrm{own}}z(p_A) + \beta_{\mathrm{opp}}z(p_B) + \beta_{\Delta}z(\Delta p) + \beta_{\mathrm{close}}z(v_{\mathrm{close}}) + \beta_{\mathrm{align}}z(A_{AB})\right)$.

The most interpretable possible results would be $HR_{\Delta p} < 1$, meaning an order advantage protects the focal run, or $HR_{\mathrm{close}} > 1$, meaning inter-team compression increases termination risk.

This analysis is exploratory because only a small subset of matches contains both tracked teams. It is useful for investigating opponent pressure and tail truncation, but it should not dominate the main paper unless the signal is very clean.

---

## 23. Current paper logic

The figures tell a sequential story.

### Figure 1: schematic

The schematic should explain the workflow visually: $\text{players} \rightarrow \text{team centroid} \rightarrow \text{centroid runs} \rightarrow \text{collective order} \rightarrow \text{termination hazard}$.

It should communicate that we are modelling team movement as a collective transport process.

### Figure 2: transport phenotype

Figure 2 shows that:

1. player and centroid runs have broad duration/length distributions;
2. centroid motion contributes substantially to player MSD;
3. team-level transport is a real object of analysis.

### Figure 3: order and transport

Figure 3 shows that:

1. high-polarisation states move faster;
2. high-order centroid runs have longer duration tails;
3. high-order centroid runs have longer length tails;
4. length differences arise from duration, speed, or both.

### Figure 4: hazard mechanism

Figure 4 shows that:

1. run termination risk decreases with run age;
2. collective order modulates termination risk;
3. state switching plus inverse-age killing can reproduce empirical duration distributions.

### Figure 5: continuous stochastic model

Figure 5 shows that:

1. polarisation can be modelled as a stochastic order variable;
2. killing depends on both age and order;
3. the resulting process reproduces observed run statistics.

---

## 24. Main interpretation

The core interpretation is $\boxed{\text{football team-centroid transport is a collective, state-dependent survival process.}}$

Long centroid runs are not simply random long displacements. They are realised during coherent collective episodes where players move in a more aligned way.

The proposed mechanism is $\text{collective order} \Rightarrow \text{lower termination hazard}$ and $\text{collective order} \Rightarrow \text{higher centroid/group speed}$.

Therefore, $\text{high order} \Rightarrow \text{longer and farther centroid transport episodes}$.

Age dependence adds an additional mechanism: $\text{fragile runs die early} \Rightarrow \text{surviving runs become progressively more stable}$.

Together, $\boxed{\text{collective order + run-age selection + state persistence} \Rightarrow \text{broad, tempered centroid-transport tails}.}$

---

## 25. Important caveats

### 1. $p_{\mathrm{mean}}$ is retrospective

Run-mean polarisation is useful for describing realised run phenotypes, but it should not be used as a purely predictive variable.

For predictive claims, use $p_{\mathrm{start}}$, $p_{\mathrm{early}}$, or $\bar p_{\mathrm{past},W}$.

### 2. Speed and order are related

High-order states often move faster. Therefore, hazard models should test whether order remains protective after speed controls.

### 3. Run segmentation depends on the turning threshold

The turning threshold $\theta_{\mathrm{max}}$ affects run durations and lengths. Main conclusions should be checked for nearby values of the threshold.

### 4. Head-to-head coupling is exploratory

Opponent coupling is scientifically interesting, but the paired-match sample is smaller. It should be presented as an exploratory extension unless the results are robust.

---

## 26. One-sentence summary

This notebook tests whether football team-centroid movement can be understood as a collective active-matter transport process in which coherent team order increases speed, suppresses run termination, and generates broad run-duration and run-length statistics.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
import s3fs
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../src"))
load_dotenv()

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches, animation
from config.plotting import configure_plotting

configure_plotting()

# ----------------------------------------------------------
# Imports (now clean via utils/__init__.py)
# ----------------------------------------------------------
from utils import (
    MatchesLoader,
    PlayerNameMapper,
    DayDataLoader,
    SoccermonMatchIndex,
)

# ----------------------------------------------------------
# S3 + loaders
# ----------------------------------------------------------
fs = s3fs.S3FileSystem(anon=False)
BUCKET = os.getenv("S3_BUCKET", "ucl-ai-soccormon-dataset")

mapper = PlayerNameMapper("src/config/player_map.json")
day_loader = DayDataLoader(fs, mapper)

# ----------------------------------------------------------
# Load match schedule (kamper.xlsx)
# ----------------------------------------------------------
matches = MatchesLoader("kamper.xlsx").load()

# ----------------------------------------------------------
# Build match index + infer which club is Source A/B
# ----------------------------------------------------------
index = SoccermonMatchIndex(
    fs=fs,
    bucket=BUCKET,
    day_loader=day_loader,
    matches=matches,
    year="2020",
    source_base_A="objective_TEAM_A",   # adjust if your folder base differs
    source_base_B="objective_team_B",
)

info = index.infer()
print(info)  # shows inferred team names + head-to-head count

### Single Match Upload

In [ ]:
# list matches involving Source A
A_matches = index.matches_for_source("A")
display(A_matches[["date","time","home","away","score"]].head(20))

gnum = 8
row = A_matches.iloc[gnum]

# load ALL parquets for that DATE (day-level) for Source A
df_raw_A = index.load_team_match(row, source_key="A", one_hz=True)   # raw, full day

print(df_raw_A.shape)
print(df_raw_A["team"].unique(), df_raw_A["source_key"].unique())
df_raw_A.head()

### Head-to-head Match Loading

In [ ]:
h2h = index.head_to_head()
display(h2h[["date","time","home","away","score"]])  # should show 2 fixtures

which = 1  # 0 or 1
row_h2h = h2h.iloc[which]
df_raw_h2h = index.load_head_to_head(which, one_hz=True)  # raw, full day for both teams

# Identify the focal team (Source A) and opposing team (Source B)
TEAM_A_NAME = df_raw_h2h.loc[df_raw_h2h["source_key"] == "A", "team"].mode().iloc[0]
TEAM_B_NAME = df_raw_h2h.loc[df_raw_h2h["source_key"] == "B", "team"].mode().iloc[0]

print(df_raw_h2h.shape)
print(df_raw_h2h["team"].value_counts())
print("TEAM_A_NAME:", TEAM_A_NAME)
print("TEAM_B_NAME:", TEAM_B_NAME)
df_raw_h2h.head()


### Transform Coordinate System onto Pitch

In [ ]:
import json
import pandas as pd

from utils.pitch_calibration import calibrate_pitch_from_df, attach_xy_from_pitch

# ---- 0) pick the head-to-head dataframe so BOTH teams are carried through ----
df_in = df_raw_h2h.copy()

# ---- 1) (recommended) ensure timestamp is datetime ----
df_in["timestamp"] = pd.to_datetime(df_in["timestamp"])

# ---- 2) load pitch polygons ----
with open("toppserien_pitches.json", "r", encoding="utf-8") as f:
    pitches = json.load(f)

# ---- 3) calibrate pitch + attach x/y (metres) ----
stadium, center_latlon, R, pitch_xy = calibrate_pitch_from_df(
    df_in, pitches, lat_col="lat", lon_col="lon"
)

df_xy_h2h = attach_xy_from_pitch(
    df_in,
    center_latlon,
    R,
    lat_col="lat",
    lon_col="lon",
    stadium_name=stadium,
)

# keep the old variable name for downstream notebook compatibility
df_xy = df_xy_h2h.copy()

print("Stadium:", stadium)
print("XY attached (both teams):", df_xy.shape)
print(df_xy["team"].value_counts())
df_xy.head()


### Identify Match Timings

In [ ]:
import pandas as pd
from utils.match_phases import label_match_phases

# ----------------------------------------------------------
# Kickoff timestamp from the selected head-to-head fixture
# ----------------------------------------------------------
kickoff_ts = pd.to_datetime(f"{row_h2h['date']} {row_h2h['time']}", errors="coerce")

# Align tz-awareness with df_xy["timestamp"]
ts = pd.to_datetime(df_xy["timestamp"])
if getattr(ts.dt, "tz", None) is not None and kickoff_ts.tzinfo is None:
    kickoff_ts = kickoff_ts.tz_localize(ts.dt.tz)
elif getattr(ts.dt, "tz", None) is None and kickoff_ts.tzinfo is not None:
    kickoff_ts = kickoff_ts.tz_convert(None)

# ----------------------------------------------------------
# Label match phases (pre, 1H, HT, 2H, post)
# ----------------------------------------------------------
df_xy = label_match_phases(df_xy, kickoff_ts)
print(df_xy["match_phase"].value_counts())

# Keep only match periods (both teams still present)
df_match_h2h = df_xy[df_xy["match_phase"].isin(["1H", "2H"])].copy()

# keep old variable name for downstream notebook compatibility
df_match = df_match_h2h.copy()

print("Match-only rows (both teams):", df_match.shape)
print(df_match["team"].value_counts())

df_match.head()


### Label Bench/Active Players within Match

In [ ]:
from utils.player_status import label_active_players

# ----------------------------------------------------------
# Active/bench labelling (keep BOTH teams through this stage)
# ----------------------------------------------------------
ACTIVE_DEPTH_M = 6.0
df_labeled = label_active_players(
    df_match, pitch_xy,
    active_depth_m=ACTIVE_DEPTH_M,
    activate_s=70.0,
    bench_off_s=110.0,
    label_col="player_status",
)

# keep old variable name for downstream notebook compatibility
df_labeled_h2h = df_labeled_h2h.copy()

print("Active/bench labelling complete.")
print(df_labeled["player_status"].value_counts())
print(df_labeled.groupby(["team", "player_status"]).size())
df_labeled.head()


In [ ]:
from viz.animation import animate_players_with_status
from IPython.display import HTML

active_depth = df_labeled.attrs.get("active_depth_m", None)

ani = animate_players_with_status(
    df_labeled,
    pitch_xy,
    matches=A_matches,        # the kamper table you loaded
    game_number=gnum+1,   # if gnum is 0-indexed in your notebook
    colour_by="status",
    show_trails=True,
    trail_only_active=True,
    trail_alpha=0.6,        # controlled from outside
    trail_s=10,
    step_s=2,
    max_frames=300,
)

HTML(ani.to_jshtml())


In [ ]:
ani.save(
    "match_animation.mp4",
    writer="ffmpeg",
    fps=25,        # change as you like
    dpi=150,       # higher = sharper, larger file
)

### Filter out Bench Players and Analyse Player Dynamics

In [ ]:

# ----------------------------------------------------------
# Keep both teams in df_active_h2h.
# Use CURRENT_TEAM as the focal dataframe for all downstream
# single-team plots/analysis, while preserving a tightly
# organised per-team structure for A and B.
# ----------------------------------------------------------
df_active = df_labeled[df_labeled["player_status"] == "active"].copy()

TEAM_NAMES = sorted(df_active["team"].astype(str).dropna().unique())
FOCAL_TEAM = TEAM_A_NAME
df_active_focal = df_active[df_active["team"].astype(str) == str(FOCAL_TEAM)].copy()

print("df_active (all active teams) shape:", df_active.shape)
print(df_active["team"].value_counts())

print("FOCAL_TEAM:", FOCAL_TEAM)
print("df_active_focal shape:", df_active_focal.shape)
print(df_active_focal["team"].value_counts())

### Set up data structurs for figures

In [ ]:
# ============================================================
# Transport tables: one source of truth for paper figures
# ============================================================

import importlib
import utils.trajectory_stats as ts
importlib.reload(ts)

# Use the v2 paper-facing transport pipeline explicitly
from utils.trajectory_stats import build_transport_tables_from_active_v2

transport = build_transport_tables_from_active_v2(
    df_active,
    theta_deg=30.0,
    max_k=120,
    max_gap_s=2.0,
    min_len=30,
    min_players_centroid=7,
    player_col="player_name",
    time_col="timestamp" if "timestamp" in df_active.columns else "_t",
    phase_col="match_phase",
    half_col="half",
    source_col="source_key" if "source_key" in df_active.columns else None,
    match_col="match_id",
    default_match_id="single_match",
    x_col="x_m",
    y_col="y_m",
    phases=("1H", "2H"),
    min_run_samples=2,
    min_disp_m=1e-6,
)

trajectory_long = transport["trajectory_long"]
msd_long = transport["msd_long"]
runs_long = transport["runs_long"]
ccdf_long = transport["ccdf_long"]
msd_decomp_long = transport["msd_decomp_long"]
figure_index = transport["figure_index"]

print("trajectory_long:", trajectory_long.shape)
print("msd_long:", msd_long.shape)
print("runs_long:", runs_long.shape)
print("ccdf_long:", ccdf_long.shape)
print("msd_decomp_long:", msd_decomp_long.shape)
print("figure_index:", figure_index.shape)

display(
    trajectory_long
    .groupby(["team", "track_type"])["track_uid"]
    .nunique()
    .reset_index(name="n_tracks")
)

display(figure_index.head(20))

### Figure 2. Transport Panel - Showing heavy tail behaviour of duration and length CCDFs + decomposition of MSD showing majority of superdiffusiveness attributable to centroid dynamics not individual players

In [ ]:
# ============================================================
# FIGURE-2 STYLE TRANSPORT PANEL
# TRUE FULL-MATCH MSD + CCDF AGGREGATION
# ============================================================
# Requires:
#   trajectory_long
#   msd_long
#   runs_long
#
# Does NOT use ccdf_long, because ccdf_long is usually per-half/path.
# Instead, whole-match CCDFs are recomputed directly from runs_long.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

OUT_DIR = Path("paper_draft_figures/figure2_transport_fullmatch_aggregated")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = False
SHOW_FIRST_N = 4
MAX_PLAYERS = None

FILTER_MATCH_ID = None       # e.g. "single_match"
FILTER_TEAM = None           # e.g. "rosenborg"
FILTER_PLAYER = None         # e.g. "Katie"

MSD_TAU_MAX_S = 120

CCDF_LENGTH_MIN_M = 0.1
CCDF_DURATION_MIN_S = 1.0

MIN_SCATTER_DURATION_S = 0.0
MIN_SCATTER_LENGTH_M = 0.0

LABELS = {
    "player_abs": "player",
    "centroid": "team centroid",
    "player_rel": "player relative to centroid",
}

LINESTYLES = {
    "player_abs": "-",
    "centroid": "-.",
    "player_rel": "--",
}

COLORS = {
    "player_abs": "C0",
    "centroid": "C1",
    "player_rel": "C2",
    "frac_centroid": "C1",
    "frac_rel": "C2",
    "frac_cross": "C3",
}

PLOT_ORDER = ["player_abs", "centroid", "player_rel"]


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_filename(s):
    return (
        str(s)
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(":", "-")
        .replace("=", "")
    )


def clean_axis(ax):
    ax.grid(alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def dedup_legend(ax, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    by_label = {}
    for h, lab in zip(handles, labels):
        if lab not in by_label and lab != "_nolegend_":
            by_label[lab] = h
    if by_label:
        ax.legend(by_label.values(), by_label.keys(), **kwargs)


def empirical_ccdf(values, min_value=None):
    """
    One clean empirical CCDF from pooled whole-match values.
    Uses unique thresholds so repeated integer durations do not make
    ugly stacked duplicate points.
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]

    if min_value is not None:
        x = x[x >= min_value]

    if len(x) == 0:
        return pd.DataFrame(columns=["threshold", "ccdf"])

    x = np.sort(x)
    n = len(x)

    thresholds, first_idx = np.unique(x, return_index=True)
    ccdf = (n - first_idx) / n

    return pd.DataFrame({
        "threshold": thresholds,
        "ccdf": ccdf,
    })


def fullmatch_base_mask(df, row):
    mask = (
        (df["match_id"] == row["match_id"]) &
        (df["team"] == row["team"])
    )

    if "source_key" in df.columns and "source_key" in row.index:
        mask &= df["source_key"].astype(str).eq(str(row["source_key"]))

    return mask


def fullmatch_entity_mask(df, row):
    return (
        (
            (df["player_name"] == row["player_name"]) &
            (df["track_type"].isin(["player_abs", "player_rel"]))
        )
        |
        (
            (df["player_name"] == "__centroid__") &
            (df["track_type"] == "centroid")
        )
    )


def combine_msd_fullmatch(msd_sel, tau_max=None):
    """
    Combines 1H + 2H MSD curves into one curve per track_type.

    Uses n_pairs weighting when available.
    Groups by track_type and k, because k is the true lag index.
    """
    if msd_sel.empty:
        return pd.DataFrame(columns=["track_type", "k", "tau_s", "msd_m2", "n_pairs"])

    d = msd_sel.copy()

    if tau_max is not None:
        d = d[d["tau_s"] <= tau_max].copy()

    d = d.replace([np.inf, -np.inf], np.nan)
    d = d.dropna(subset=["track_type", "k", "tau_s", "msd_m2"])

    if d.empty:
        return pd.DataFrame(columns=["track_type", "k", "tau_s", "msd_m2", "n_pairs"])

    if "n_pairs" not in d.columns:
        d["n_pairs"] = 1.0

    d["n_pairs"] = d["n_pairs"].fillna(1.0).astype(float)
    d = d[d["n_pairs"] > 0].copy()

    d["tau_w"] = d["tau_s"] * d["n_pairs"]
    d["msd_w"] = d["msd_m2"] * d["n_pairs"]

    out = (
        d.groupby(["track_type", "k"], as_index=False)
         .agg(
             tau_w=("tau_w", "sum"),
             msd_w=("msd_w", "sum"),
             n_pairs=("n_pairs", "sum"),
         )
    )

    out["tau_s"] = out["tau_w"] / out["n_pairs"]
    out["msd_m2"] = out["msd_w"] / out["n_pairs"]

    return (
        out[["track_type", "k", "tau_s", "msd_m2", "n_pairs"]]
        .sort_values(["track_type", "k"])
        .reset_index(drop=True)
    )


def build_wholematch_ccdf_from_runs(runs_sel):
    """
    Recompute whole-match CCDFs from pooled 1H+2H runs.
    This avoids duplicated per-half CCDF curves.
    """
    rows = []

    for track_type, g in runs_sel.groupby("track_type", sort=False):
        if track_type not in PLOT_ORDER:
            continue

        d_len = empirical_ccdf(g["run_length_m"], min_value=CCDF_LENGTH_MIN_M)
        d_len["track_type"] = track_type
        d_len["variable"] = "run_length_m"
        rows.append(d_len)

        d_dur = empirical_ccdf(g["duration_s"], min_value=CCDF_DURATION_MIN_S)
        d_dur["track_type"] = track_type
        d_dur["variable"] = "duration_s"
        rows.append(d_dur)

    if len(rows) == 0:
        return pd.DataFrame(columns=["threshold", "ccdf", "track_type", "variable"])

    return pd.concat(rows, ignore_index=True)


def build_fractional_msd_decomp(msd_combined):
    """
    Builds fractional MSD decomposition from already-combined MSD.

    Identity:
        MSD_abs = MSD_centroid + MSD_rel + cross

    Fractions:
        f_centroid = MSD_centroid / MSD_abs
        f_rel      = MSD_rel / MSD_abs
        f_cross    = cross / MSD_abs
    """
    if msd_combined.empty:
        return pd.DataFrame()

    wide = (
        msd_combined
        .pivot_table(index="k", columns="track_type", values=["tau_s", "msd_m2"], aggfunc="mean")
    )

    # Flatten columns
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()

    required = [
        "msd_m2_player_abs",
        "msd_m2_centroid",
        "msd_m2_player_rel",
    ]
    missing = [c for c in required if c not in wide.columns]
    if missing:
        return pd.DataFrame()

    # Use player_abs tau as the x-axis if available
    if "tau_s_player_abs" in wide.columns:
        wide["tau_s"] = wide["tau_s_player_abs"]
    else:
        tau_cols = [c for c in wide.columns if c.startswith("tau_s_")]
        wide["tau_s"] = wide[tau_cols].mean(axis=1)

    wide = wide.dropna(subset=required + ["tau_s"]).copy()

    if wide.empty:
        return pd.DataFrame()

    wide["cross_msd"] = (
        wide["msd_m2_player_abs"]
        - wide["msd_m2_centroid"]
        - wide["msd_m2_player_rel"]
    )

    denom = wide["msd_m2_player_abs"].replace(0, np.nan)

    wide["frac_centroid"] = wide["msd_m2_centroid"] / denom
    wide["frac_rel"] = wide["msd_m2_player_rel"] / denom
    wide["frac_cross"] = wide["cross_msd"] / denom

    # Keep finite values only
    for c in ["frac_centroid", "frac_rel", "frac_cross"]:
        wide[c] = wide[c].replace([np.inf, -np.inf], np.nan)

    return wide.sort_values("tau_s").reset_index(drop=True)


def get_fullmatch_view(row):
    """
    Returns:
      traj_sel: still phase/path separated for sensible trajectory drawing
      msd_combined: one whole-match MSD curve per track_type
      runs_sel: pooled full-match runs
      ccdf_combined: one whole-match CCDF per track_type and variable
      frac_df: fractional MSD decomposition
    """
    traj_sel = trajectory_long[
        fullmatch_base_mask(trajectory_long, row) &
        fullmatch_entity_mask(trajectory_long, row)
    ].copy()

    msd_sel_raw = msd_long[
        fullmatch_base_mask(msd_long, row) &
        fullmatch_entity_mask(msd_long, row)
    ].copy()

    runs_sel = runs_long[
        fullmatch_base_mask(runs_long, row) &
        fullmatch_entity_mask(runs_long, row)
    ].copy()

    msd_combined = combine_msd_fullmatch(msd_sel_raw, tau_max=MSD_TAU_MAX_S)
    ccdf_combined = build_wholematch_ccdf_from_runs(runs_sel)
    frac_df = build_fractional_msd_decomp(msd_combined)

    return traj_sel, msd_combined, runs_sel, ccdf_combined, frac_df


# ------------------------------------------------------------
# Candidate list: one row per player/team/source/full match
# ------------------------------------------------------------

traj_players = trajectory_long[
    (trajectory_long["track_type"] == "player_abs") &
    (trajectory_long["player_name"] != "__centroid__")
].copy()

candidate_cols = ["match_id", "team", "player_name"]
if "source_key" in traj_players.columns:
    candidate_cols.append("source_key")

candidates = (
    traj_players
    .groupby(candidate_cols, dropna=False, as_index=False)
    .agg(
        n_frames=("t", "size"),
        t_start=("t", "min"),
        t_end=("t", "max"),
    )
)

# Add run counts for sorting
run_players = runs_long[
    (runs_long["track_type"] == "player_abs") &
    (runs_long["player_name"] != "__centroid__")
].copy()

run_summary_cols = candidate_cols.copy()

run_summary = (
    run_players
    .groupby(run_summary_cols, dropna=False, as_index=False)
    .agg(
        n_runs=("run_length_m", "size"),
        median_run_length_m=("run_length_m", "median"),
        max_run_length_m=("run_length_m", "max"),
        median_duration_s=("duration_s", "median"),
        max_duration_s=("duration_s", "max"),
    )
)

candidates = candidates.merge(run_summary, on=run_summary_cols, how="left")

if FILTER_MATCH_ID is not None:
    candidates = candidates[candidates["match_id"] == FILTER_MATCH_ID].copy()

if FILTER_TEAM is not None:
    candidates = candidates[candidates["team"] == FILTER_TEAM].copy()

if FILTER_PLAYER is not None:
    candidates = candidates[candidates["player_name"] == FILTER_PLAYER].copy()

candidates = candidates.sort_values(
    ["match_id", "team", "n_frames", "max_run_length_m", "player_name"],
    ascending=[True, True, False, False, True],
).reset_index(drop=True)

if MAX_PLAYERS is not None:
    candidates = candidates.head(MAX_PLAYERS).copy()

print(f"Making TRUE full-match aggregated panels for {len(candidates)} player entries")
display(candidates.head(20))


# ------------------------------------------------------------
# Main plotting loop
# ------------------------------------------------------------

saved_paths = []

for i, row in candidates.iterrows():

    match_id = row["match_id"]
    team = row["team"]
    player = row["player_name"]
    source = row["source_key"] if "source_key" in row.index else ""

    traj_sel, msd_combined, runs_sel, ccdf_combined, frac_df = get_fullmatch_view(row)

    if traj_sel.empty or msd_combined.empty or runs_sel.empty:
        continue

    runs_scatter = runs_sel[
        (runs_sel["duration_s"] >= MIN_SCATTER_DURATION_S) &
        (runs_sel["run_length_m"] >= MIN_SCATTER_LENGTH_M)
    ].copy()

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    ax_traj, ax_msd, ax_len = axes[0]
    ax_dur, ax_lt, ax_frac = axes[1]

    # ========================================================
    # A. Trajectory: player absolute + centroid
    # ========================================================

    for track_type in ["player_abs", "centroid"]:
        g = traj_sel[traj_sel["track_type"] == track_type].sort_values("t")
        if g.empty:
            continue

        group_cols = []
        if "match_phase" in g.columns:
            group_cols.append("match_phase")
        if "path_uid" in g.columns:
            group_cols.append("path_uid")

        groups = g.groupby(group_cols, sort=False) if group_cols else [("all", g)]

        first = True
        for _, gp in groups:
            ax_traj.plot(
                gp["x_m"],
                gp["y_m"],
                lw=1.2 if track_type == "player_abs" else 2.0,
                alpha=0.75 if track_type == "player_abs" else 0.95,
                linestyle=LINESTYLES.get(track_type, "-"),
                color=COLORS.get(track_type, None),
                label=LABELS.get(track_type, track_type) if first else None,
            )
            first = False

    ax_traj.set_title("A. trajectory")
    ax_traj.set_xlabel("x [m]")
    ax_traj.set_ylabel("y [m]")
    ax_traj.set_aspect("equal", adjustable="box")
    clean_axis(ax_traj)
    dedup_legend(ax_traj, fontsize=8, loc="best")

    # ========================================================
    # B. Combined full-match MSD
    # ========================================================

    for track_type in PLOT_ORDER:
        g = msd_combined[msd_combined["track_type"] == track_type].sort_values("tau_s")
        if g.empty:
            continue

        ax_msd.plot(
            g["tau_s"],
            g["msd_m2"],
            lw=2.2,
            linestyle=LINESTYLES.get(track_type, "-"),
            color=COLORS.get(track_type, None),
            label=LABELS.get(track_type, track_type),
        )

    ax_msd.set_xscale("log")
    ax_msd.set_yscale("log")
    ax_msd.set_title("B. MSD")
    ax_msd.set_xlabel("lag time [s]")
    ax_msd.set_ylabel(r"MSD [m$^2$]")
    clean_axis(ax_msd)
    dedup_legend(ax_msd, fontsize=8, loc="best")

    # ========================================================
    # C. Whole-match run-length CCDF
    # ========================================================

    for track_type in PLOT_ORDER:
        g = ccdf_combined[
            (ccdf_combined["track_type"] == track_type) &
            (ccdf_combined["variable"] == "run_length_m")
        ].sort_values("threshold")

        if g.empty:
            continue

        ax_len.plot(
            g["threshold"],
            g["ccdf"],
            lw=2.2,
            linestyle=LINESTYLES.get(track_type, "-"),
            color=COLORS.get(track_type, None),
            label=LABELS.get(track_type, track_type),
        )

    ax_len.set_xscale("log")
    ax_len.set_yscale("log")
    ax_len.set_title("C. run-length CCDF")
    ax_len.set_xlabel("run length [m]")
    ax_len.set_ylabel(r"$P(L \geq \ell)$")
    clean_axis(ax_len)
    dedup_legend(ax_len, fontsize=8, loc="best")

    # ========================================================
    # D. Whole-match run-duration CCDF
    # ========================================================

    for track_type in PLOT_ORDER:
        g = ccdf_combined[
            (ccdf_combined["track_type"] == track_type) &
            (ccdf_combined["variable"] == "duration_s")
        ].sort_values("threshold")

        if g.empty:
            continue

        ax_dur.plot(
            g["threshold"],
            g["ccdf"],
            lw=2.2,
            linestyle=LINESTYLES.get(track_type, "-"),
            color=COLORS.get(track_type, None),
            label=LABELS.get(track_type, track_type),
        )

    ax_dur.set_xscale("log")
    ax_dur.set_yscale("log")
    ax_dur.set_title("D. run-duration CCDF")
    ax_dur.set_xlabel("duration [s]")
    ax_dur.set_ylabel(r"$P(T \geq t)$")
    clean_axis(ax_dur)
    dedup_legend(ax_dur, fontsize=8, loc="best")

    # ========================================================
    # E. L vs T
    # ========================================================

    for track_type in PLOT_ORDER:
        g = runs_scatter[runs_scatter["track_type"] == track_type].copy()
        if g.empty:
            continue

        ax_lt.scatter(
            g["duration_s"],
            g["run_length_m"],
            s=10 if track_type != "centroid" else 18,
            alpha=0.30 if track_type != "centroid" else 0.65,
            color=COLORS.get(track_type, None),
            label=LABELS.get(track_type, track_type),
        )

    ax_lt.set_xscale("log")
    ax_lt.set_yscale("log")
    ax_lt.set_title(r"E. $L$ vs $T$")
    ax_lt.set_xlabel("duration [s]")
    ax_lt.set_ylabel("run length [m]")
    clean_axis(ax_lt)
    dedup_legend(ax_lt, fontsize=8, loc="best")

    # ========================================================
    # F. Fractional MSD decomposition
    # ========================================================

    if not frac_df.empty:
        ax_frac.plot(
            frac_df["tau_s"],
            frac_df["frac_centroid"],
            lw=2.2,
            color=COLORS["frac_centroid"],
            label="centroid fraction",
        )
        ax_frac.plot(
            frac_df["tau_s"],
            frac_df["frac_rel"],
            lw=2.2,
            color=COLORS["frac_rel"],
            label="relative fraction",
        )
        ax_frac.plot(
            frac_df["tau_s"],
            frac_df["frac_cross"],
            lw=2.2,
            color=COLORS["frac_cross"],
            label="cross fraction",
        )

        ax_frac.axhline(0.0, color="k", lw=0.8, alpha=0.6)
        ax_frac.axhline(1.0, color="k", lw=0.8, alpha=0.25, linestyle=":")

        ax_frac.set_xscale("log")
        ax_frac.set_title("F. MSD fractional decomposition")
        ax_frac.set_xlabel("lag time [s]")
        ax_frac.set_ylabel("fraction of player MSD")

        clean_axis(ax_frac)
        dedup_legend(ax_frac, fontsize=8, loc="best")

    else:
        ax_frac.text(
            0.5,
            0.5,
            "No decomposition available",
            ha="center",
            va="center",
            transform=ax_frac.transAxes,
        )
        ax_frac.set_title("F. MSD fractional decomposition")
        clean_axis(ax_frac)

    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    fig.suptitle(
        f"{match_id} | full match | {team} | {player}",
        y=0.98,
        fontsize=13,
        fontweight="bold",
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    # --------------------------------------------------------
    # Save/show
    # --------------------------------------------------------

    fname = (
        f"transport_panel_TRUE_fullmatch"
        f"__match-{safe_filename(match_id)}"
        f"__team-{safe_filename(team)}"
        f"__source-{safe_filename(source)}"
        f"__player-{safe_filename(player)}.png"
    )
    out_path = OUT_DIR / fname

    if SAVE_FIGS:
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        saved_paths.append(out_path)

    if i < SHOW_FIRST_N:
        plt.show()
    else:
        plt.close(fig)

print(f"Saved {len(saved_paths)} figures to: {OUT_DIR}")
if saved_paths:
    print("First few saved files:")
    for p in saved_paths[:10]:
        print(" ", p)

In [ ]:
import importlib
import utils.trajectory_stats as ts
importlib.reload(ts)

heading_runs = ts.prepare_heading_runs_from_transport(
    transport,
    match_id="single_match",
    team=None,
    source_key=None,
    track_type="player_abs",
    combine_halves=True,
    min_run_length_m=0.0,
)

A_df = ts.anisotropy_by_player_transport(
    heading_runs,
    player_cols=["player_name"],   # players are unique in your data
    weight_col="run_length_m",
    B=500,
    seed=1,
)

A_plot = A_df.sort_values("A_w", ascending=False).reset_index(drop=True)

display(heading_runs.head())
display(A_plot.head(20))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Heading anisotropy figure from new transport style
# ============================================================

player_for_panelA = "Abigail"

heading_col = "heading_combined_rad"
weight_col = "run_length_m"
n_angle_bins = 24
min_step_length = 0.0

g = heading_runs[heading_runs["player_name"] == player_for_panelA].copy()

theta = g[heading_col].to_numpy()
L = g[weight_col].to_numpy()

m = np.isfinite(theta) & np.isfinite(L)
theta = theta[m]
L = L[m]

mask_w = L >= min_step_length
theta_w = theta[mask_w]
L_w = L[mask_w]

edges = np.linspace(-np.pi, np.pi, n_angle_bins + 1)
centers = 0.5 * (edges[:-1] + edges[1:])
widths = np.diff(edges)

counts, _ = np.histogram(theta, bins=edges)
p = counts / counts.sum() if counts.sum() > 0 else np.zeros_like(counts, dtype=float)

wcounts, _ = np.histogram(theta_w, bins=edges, weights=L_w)
pw = wcounts / wcounts.sum() if wcounts.sum() > 0 else np.zeros_like(wcounts, dtype=float)

fig, (axA, axB) = plt.subplots(
    2,
    1,
    figsize=(8, 6),
    gridspec_kw={"height_ratios": [3, 3]},
    constrained_layout=True,
)

# =========================
# Panel A: heading distribution
# =========================

axA.bar(
    centers,
    p,
    width=widths,
    align="center",
    alpha=0.9,
    facecolor="b",
    edgecolor="k",
    linewidth=1,
    label="Unweighted",
)

axA.bar(
    centers,
    pw,
    width=widths,
    align="center",
    alpha=0.45,
    facecolor="r",
    edgecolor="k",
    linewidth=1,
    label="Length-weighted",
)

axA.set_xlim(-np.pi, np.pi)
axA.set_xlabel(r"Run heading $\theta$ [rad]")
axA.set_ylabel("Probability")
axA.set_title(f"Heading distribution for {player_for_panelA}")

xticks = [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi]
xtick_labels = [r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"]
axA.set_xticks(xticks)
axA.set_xticklabels(xtick_labels)

axA.legend(frameon=False, ncol=2)

# =========================
# Panel B: all-player axial anisotropy
# =========================

x = np.arange(len(A_plot))
labels = A_plot["player_name"].astype(str).to_list()

y = A_plot["A_w"].to_numpy()
yerr = np.vstack([
    y - A_plot["A_w_lo"].to_numpy(),
    A_plot["A_w_hi"].to_numpy() - y,
])

axB.errorbar(
    x,
    y,
    yerr=yerr,
    c="k",
    fmt=".",
    capsize=3,
    label="Length-weighted",
)

y2 = A_plot["A_unw"].to_numpy()
yerr2 = np.vstack([
    y2 - A_plot["A_unw_lo"].to_numpy(),
    A_plot["A_unw_hi"].to_numpy() - y2,
])

axB.errorbar(
    x,
    y2,
    yerr=yerr2,
    c="r",
    fmt=".",
    capsize=3,
    label="Unweighted",
    alpha=0.85,
)

axB.axhline(0, lw=1, c="k")
axB.set_xticks(x)
axB.set_xticklabels(labels, rotation=65)
axB.set_ylabel(r"$\langle \cos(2\theta)\rangle$")
axB.set_title(r"Axial alignment across players")
axB.legend(frameon=False)

plt.show()

#### Supplementary Figure. Directional anisotropy of transport and pitch-axis alignment 

### Collective Behaviour - Linking Dynamics and Polarisation

In [ ]:
# ============================================================
# COLLECTIVE ORDER TABLES FROM transport
# Keep BOTH teams by default
# ============================================================

import importlib
import pandas as pd
import numpy as np
import utils.collective_stats as cs

importlib.reload(cs)

# Choose how centroid runs are split into low/mid/high order states.
# "p_mean" is descriptive and often separates CCDFs clearly.
# "p_early_3s" is cleaner as a near-predictor.
# "p_start" is strictest but noisier.
ORDER_STATE_COL = "p_mean"

# ------------------------------------------------------------
# 1) Build collective order + centroid-run state tables
# ------------------------------------------------------------

df_pmv, centroid_order_runs = cs.build_collective_order_tables_from_transport(
    transport,
    order_state_col=ORDER_STATE_COL,
    early_window_s=3.0,
    min_speed_mps=0.0,
    team_col="team",
    phase_col="match_phase",
    player_col="player_name",
    path_col="path_uid",
    t_col="_t",
    x_col="x_m",
    y_col="y_m",
    phases=("1H", "2H"),
    verbose=True,
)

# Backward-compatible alias for two-team analyses.
# IMPORTANT: this is BOTH teams.
df_pmv_h2h = df_pmv.copy()

# ------------------------------------------------------------
# 2) Optional polarisation-only table, also BOTH teams
# ------------------------------------------------------------

df_pol_h2h = cs.build_df_polarisation(
    transport["df_paths"],
    min_speed_mps=0.5,
    team_col="team",
    phase_col="match_phase",
    player_col="player_name",
    path_col="path_uid",
    t_col="_t",
    x_col="x_m",
    y_col="y_m",
    phases=("1H", "2H"),
).copy()

# Keep df_pol as both-team table too.
df_pol = df_pol_h2h.copy()

# ------------------------------------------------------------
# 3) Optional focal-team aliases for old cells only
# ------------------------------------------------------------

FOCAL_TEAM = sorted(df_pmv["team"].astype(str).unique())[0]

df_pmv_focal = df_pmv_h2h[df_pmv_h2h["team"].astype(str) == FOCAL_TEAM].copy()
df_pol_focal = df_pol_h2h[df_pol_h2h["team"].astype(str) == FOCAL_TEAM].copy()

centroid_ts_h2h = transport["centroid_ts"].copy()
centroid_ts_focal = centroid_ts_h2h[
    centroid_ts_h2h["team"].astype(str) == FOCAL_TEAM
].copy()

# ------------------------------------------------------------
# 4) Diagnostics
# ------------------------------------------------------------

print("=" * 100)
print("df_pmv / df_pmv_h2h teams")
print("=" * 100)
display(
    df_pmv
    .groupby(["team", "match_phase"], observed=True)
    .size()
    .reset_index(name="n")
)

print("=" * 100)
print("centroid_order_runs states")
print("=" * 100)
display(
    centroid_order_runs
    .groupby(["team", "match_phase", "p_state"], observed=True, dropna=False)
    .size()
    .reset_index(name="n_runs")
)

print("=" * 100)
print("df_pol_h2h teams")
print("=" * 100)
display(
    df_pol_h2h
    .groupby(["team", "match_phase"], observed=True)
    .size()
    .reset_index(name="n")
)

print("ORDER_STATE_COL:", ORDER_STATE_COL)
print("Teams in df_pmv:", sorted(df_pmv["team"].astype(str).unique()))
print("Teams in centroid_order_runs:", sorted(centroid_order_runs["team"].astype(str).unique()))
print("FOCAL_TEAM alias only:", FOCAL_TEAM)

### Fig. 3 Collective order linked to centroid speed. CCDFs are split substantially by mean collective order via mean polarisation

In [ ]:
# ============================================================
# FIGURE 3 PLOT: collective order and centroid transport
# One 4-panel figure per team
# ============================================================
# Inputs:
#   df_pmv
#   centroid_order_runs
#
# Assumes centroid_order_runs was built with:
#   cs.build_collective_order_tables_from_transport(...)
#
# Required columns:
#   df_pmv: team, match_phase, _t, p_group, v_group_mps
#   centroid_order_runs: team, p_state, duration_s, run_length_m
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

OUT_DIR = Path("paper_draft_figures/figure3_collective_order")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = False
SHOW_FIGS = True

# Must match the variable used when centroid_order_runs was built.
# e.g. "p_start", "p_early_3s", "p_mean", "p_median"
ORDER_STATE_COL = (
    centroid_order_runs["p_state_source"]
    .dropna()
    .astype(str)
    .iloc[0]
)

MIN_RUN_DURATION_S = 1.0
MIN_RUN_LENGTH_M = 0.1

STATE_ORDER = ["low", "mid", "high"]

STATE_COLORS = {
    "low": "C0",
    "mid": "C1",
    "high": "C3",
}

STATE_LABELS = {
    "low": "low polarisation",
    "mid": "mid polarisation",
    "high": "high polarisation",
}

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_filename(s):
    return (
        str(s)
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(":", "-")
        .replace("=", "")
    )


def empirical_ccdf(values, min_value=None):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]

    if min_value is not None:
        x = x[x >= min_value]

    if len(x) == 0:
        return np.array([]), np.array([])

    x = np.sort(x)
    n = len(x)

    thresholds, first_idx = np.unique(x, return_index=True)
    ccdf = (n - first_idx) / n

    return thresholds, ccdf


def add_match_clock_minutes(df, time_col="_t", phase_col="match_phase"):
    """
    Continuous match clock:
      1H starts at 0 min
      2H starts at 45 min
    """
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col])
    d["match_min"] = np.nan

    phase_offsets = {
        "1H": 0.0,
        "2H": 45.0,
    }

    for phase, offset in phase_offsets.items():
        m = d[phase_col].astype(str).eq(phase)
        if not m.any():
            continue

        t0 = d.loc[m, time_col].min()
        d.loc[m, "match_min"] = (
            (d.loc[m, time_col] - t0).dt.total_seconds() / 60.0 + offset
        )

    return d


def clean_axis(ax):
    ax.grid(alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def state_threshold_text(runs_team, order_col):
    """
    Build a small text label showing q1/q2 thresholds if available.
    """
    if "p_q1" not in runs_team.columns or "p_q2" not in runs_team.columns:
        return ""

    q1 = pd.to_numeric(runs_team["p_q1"], errors="coerce").dropna()
    q2 = pd.to_numeric(runs_team["p_q2"], errors="coerce").dropna()

    if len(q1) == 0 or len(q2) == 0:
        return ""

    return f"{order_col} terciles: q1={q1.iloc[0]:.2f}, q2={q2.iloc[0]:.2f}"


def plot_figure3_for_team(team, df_pmv, centroid_order_runs):
    """
    One 2x2 Figure 3 draft for one team.
    """

    pmv_team = df_pmv[df_pmv["team"].astype(str) == str(team)].copy()

    runs_team = centroid_order_runs[
        centroid_order_runs["team"].astype(str) == str(team)
    ].copy()

    if pmv_team.empty:
        print(f"Skipping {team}: no df_pmv rows.")
        return None

    if runs_team.empty:
        print(f"Skipping {team}: no centroid_order_runs rows.")
        return None

    # Ensure state column exists and is usable
    if "p_state" not in runs_team.columns:
        raise KeyError("centroid_order_runs must contain 'p_state'. Rebuild the Figure 3 data cell first.")

    # Keep only runs with assigned low/mid/high state
    runs_team = runs_team.dropna(subset=["p_state"]).copy()

    if runs_team.empty:
        print(f"Skipping {team}: no centroid runs with assigned p_state.")
        return None

    pmv_team = add_match_clock_minutes(pmv_team)
    pmv_team = pmv_team.sort_values("match_min").reset_index(drop=True)

    print("\n" + "=" * 100)
    print(f"Figure 3 for team: {team}")
    print("=" * 100)
    print("pmv_team:", pmv_team.shape)
    print("runs_team with p_state:", runs_team.shape)
    print(state_threshold_text(runs_team, ORDER_STATE_COL))

    display(
        runs_team
        .groupby("p_state", observed=True)
        .agg(
            n_runs=("run_uid", "size"),
            mean_p=(ORDER_STATE_COL, "mean"),
            median_duration_s=("duration_s", "median"),
            median_length_m=("run_length_m", "median"),
            max_duration_s=("duration_s", "max"),
            max_length_m=("run_length_m", "max"),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    ax_ts, ax_scatter, ax_dur, ax_len = axes.ravel()

    # ========================================================
    # A. p_group(t) and v_group(t)
    # ========================================================

    ax_ts.plot(
        pmv_team["match_min"],
        pmv_team["p_group"],
        color="C0",
        lw=1.6,
        label=r"polarisation $p(t)$",
    )

    ax_ts.set_xlabel("match time [min]")
    ax_ts.set_ylabel(r"polarisation $p(t)$", color="C0")
    ax_ts.tick_params(axis="y", labelcolor="C0")
    ax_ts.set_ylim(-0.02, 1.02)

    ax_ts2 = ax_ts.twinx()

    ax_ts2.plot(
        pmv_team["match_min"],
        pmv_team["v_group_mps"],
        color="C1",
        lw=1.2,
        alpha=0.85,
        label="group speed",
    )

    ax_ts2.set_ylabel("group speed [m/s]", color="C1")
    ax_ts2.tick_params(axis="y", labelcolor="C1")

    ax_ts.set_title("A. collective order and speed over time")
    ax_ts.grid(alpha=0.25)
    ax_ts.spines["top"].set_visible(False)
    ax_ts2.spines["top"].set_visible(False)

    h1, l1 = ax_ts.get_legend_handles_labels()
    h2, l2 = ax_ts2.get_legend_handles_labels()
    ax_ts.legend(h1 + h2, l1 + l2, frameon=False, loc="upper right", fontsize=8)

    # ========================================================
    # B. p_group vs group speed
    # ========================================================

    ax_scatter.scatter(
        pmv_team["p_group"],
        pmv_team["v_group_mps"],
        s=10,
        alpha=0.20,
        color="k",
        label="frames",
    )

    tmp = pmv_team[["p_group", "v_group_mps"]].dropna().copy()

    if len(tmp) >= 20 and tmp["p_group"].nunique() >= 5:
        tmp["p_bin"] = pd.qcut(tmp["p_group"], q=10, duplicates="drop")

        bin_stats = (
            tmp.groupby("p_bin", observed=True)
            .agg(
                p_mid=("p_group", "mean"),
                v_mean=("v_group_mps", "mean"),
                v_sem=(
                    "v_group_mps",
                    lambda x: np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan,
                ),
                n=("v_group_mps", "size"),
            )
            .reset_index()
        )

        ax_scatter.errorbar(
            bin_stats["p_mid"],
            bin_stats["v_mean"],
            yerr=bin_stats["v_sem"],
            color="C3",
            marker="o",
            lw=2,
            capsize=3,
            label="binned mean",
        )

    ax_scatter.set_xlabel(r"polarisation $p$")
    ax_scatter.set_ylabel("group speed [m/s]")
    ax_scatter.set_title("B. ordered states move faster")
    ax_scatter.legend(frameon=False)
    clean_axis(ax_scatter)

    # ========================================================
    # C. Centroid run-duration CCDF by order state
    # ========================================================

    for state in STATE_ORDER:
        g = runs_team[runs_team["p_state"].astype(str) == state].copy()

        if g.empty:
            continue

        x, y = empirical_ccdf(g["duration_s"], min_value=MIN_RUN_DURATION_S)

        if len(x) == 0:
            continue

        ax_dur.plot(
            x,
            y,
            lw=2.2,
            color=STATE_COLORS[state],
            label=f"{STATE_LABELS[state]} (n={len(g)})",
        )

    ax_dur.set_xscale("log")
    ax_dur.set_yscale("log")
    ax_dur.set_xlabel("centroid-run duration [s]")
    ax_dur.set_ylabel(r"$P(T \geq t)$")
    ax_dur.set_title(f"C. duration tails by {ORDER_STATE_COL}")
    ax_dur.legend(frameon=False, fontsize=8)
    clean_axis(ax_dur)

    # ========================================================
    # D. Centroid run-length CCDF by order state
    # ========================================================

    for state in STATE_ORDER:
        g = runs_team[runs_team["p_state"].astype(str) == state].copy()

        if g.empty:
            continue

        x, y = empirical_ccdf(g["run_length_m"], min_value=MIN_RUN_LENGTH_M)

        if len(x) == 0:
            continue

        ax_len.plot(
            x,
            y,
            lw=2.2,
            color=STATE_COLORS[state],
            label=f"{STATE_LABELS[state]} (n={len(g)})",
        )

    ax_len.set_xscale("log")
    ax_len.set_yscale("log")
    ax_len.set_xlabel("centroid-run length [m]")
    ax_len.set_ylabel(r"$P(L \geq \ell)$")
    ax_len.set_title(f"D. length tails by {ORDER_STATE_COL}")
    ax_len.legend(frameon=False, fontsize=8)
    clean_axis(ax_len)

    # --------------------------------------------------------
    # Title / save / show
    # --------------------------------------------------------

    subtitle = state_threshold_text(runs_team, ORDER_STATE_COL)
    title = f"Figure 3 draft — collective order and centroid transport | {team}"

    if subtitle:
        title += f"\n{subtitle}"

    fig.suptitle(
        title,
        fontsize=14,
        fontweight="bold",
        y=1.03,
    )

    fig.tight_layout()

    out_path = OUT_DIR / (
        f"figure3_collective_order"
        f"__team-{safe_filename(team)}"
        f"__state-{safe_filename(ORDER_STATE_COL)}.png"
    )

    if SAVE_FIGS:
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        print("Saved:", out_path)

    if SHOW_FIGS:
        plt.show()
    else:
        plt.close(fig)

    return fig


# ------------------------------------------------------------
# Plot one 4-panel figure per team
# ------------------------------------------------------------

available_teams = sorted(df_pmv["team"].dropna().astype(str).unique())
print("Available teams:", available_teams)

figs = {}

for team in available_teams:
    fig = plot_figure3_for_team(team, df_pmv, centroid_order_runs)
    figs[team] = fig

### Hazard Modelling

In [ ]:
# ============================================================
# FIGURE 4 DATA: centroid-run hazard model
# ============================================================

import importlib
import utils.hazard_stats as hs
importlib.reload(hs)

# Hazard states are based on CURRENT p_group, not p_mean.
HAZARD_STATE_COL = "p_state_current"

hazard_intervals = hs.build_centroid_hazard_intervals(
    transport,
    df_pmv,
    interval_s=1.0,
    min_duration_s=1.0,
    team_col="team",
    phase_col="match_phase",
    t_col="_t",
    p_col="p_group",
    state_col=HAZARD_STATE_COL,
    assign_states=True,
    verbose=True,
)

hazard_empirical = hs.build_empirical_hazard(
    hazard_intervals,
    state_col=HAZARD_STATE_COL,
    age_bin_width_s=2.0,
    max_age_s=60.0,
    min_exposure_s=8.0,
)

hazard_fit_summary = hs.fit_inverse_age_hazard_by_group(
    hazard_intervals,
    state_col=HAZARD_STATE_COL,
    group_cols=("team", HAZARD_STATE_COL),
    a0=1.0,
    min_events=20,
    max_age_s=60.0,
    verbose=True,
)

hazard_pred_grid = hs.build_hazard_prediction_grid(
    hazard_fit_summary,
    state_col=HAZARD_STATE_COL,
    age_max_s=60.0,
    n_grid=300,
)

hazard_run_survival = hs.build_run_survival_from_intervals(
    hazard_intervals,
    state_col=HAZARD_STATE_COL,
    min_duration_s=1.0,
)

print("=" * 100)
print("hazard_intervals")
print("=" * 100)
print(hazard_intervals.shape)
display(
    hazard_intervals
    .groupby(["team", HAZARD_STATE_COL], observed=True)
    .agg(
        n_intervals=("run_uid", "size"),
        n_runs=("run_uid", "nunique"),
        events=("event", "sum"),
        exposure_s=("dt_s", "sum"),
        mean_p=("p_group", "mean"),
    )
    .reset_index()
)

print("=" * 100)
print("hazard_empirical")
print("=" * 100)
display(hazard_empirical.head())

print("=" * 100)
print("hazard_fit_summary")
print("=" * 100)
display(hazard_fit_summary)

print("=" * 100)
print("hazard_run_survival")
print("=" * 100)
display(hazard_run_survival.head())

### Fig. 4. Hazard/Termination prob of run decreases with age. Modulated vertically by collective order. State switching + inverse age hazard enough to reproduce CCDFs?

In [ ]:
# ============================================================
# FIGURE 4 PANEL:
# Age memory + collective-state switching + killed-process model
# ============================================================
#
# Assumes the previous block created:
#   hazard_intervals
#   hazard_empirical
#   hazard_fit_summary
#   hazard_pred_grid
#   hazard_run_survival
#
# This block creates:
#   state_interval_table
#   state_transition_probs
#   run_switch_table
#   run_switch_summary
#   state_killing_fits
#   sim_runs_fig4
#   emp_runs_fig4
#   fig4_model_summary
#   fig4_panel_by_team
#
# Figure panels:
#   A. Overall hazard vs age
#   B. Hazard split by current p-state
#   C. Transition matrix
#   D. Duration CCDF by switch count
#   E. Total CCDF: empirical vs Markov-killed model
#   F. p_mean CCDF: empirical vs Markov-killed model
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

STATE_ORDER = ["low", "mid", "high"]
STATE_TO_I = {s: i for i, s in enumerate(STATE_ORDER)}
I_TO_STATE = {i: s for s, i in STATE_TO_I.items()}

STATE_COLORS = {
    "low": "tab:blue",
    "mid": "tab:orange",
    "high": "tab:red",
}

SWITCH_COLORS = {
    "0": "tab:blue",
    "1": "tab:orange",
    "2": "tab:green",
    "3+": "tab:red",
}

A0 = 1.0
DT_SIM = 1.0
MAX_T_SIM = 120.0
N_SIM_PER_TEAM = 20000
SEED = 7

AGE_BINS = np.array([1, 2, 3, 4, 7, 10, 15, 22, 32, 50, 60], dtype=float)
MIN_EXPOSURE_PER_BIN = 8

FIG_DPI = 180
SAVE_FIGURES = True
FIG_OUTDIR = "paper_figures_v3_improved"
FIG_BASENAME = "fig4_mechanism_markov_killed_process"

rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 7,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def _pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of these columns found: {candidates}")
    return None


def _ensure_outdir(path):
    if not SAVE_FIGURES:
        return
    import os
    os.makedirs(path, exist_ok=True)


def _ccdf(values):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return np.array([]), np.array([])
    x = np.sort(x)
    y = 1.0 - np.arange(len(x)) / len(x)
    return x, y


def _binom_se(k, n):
    if n <= 0:
        return np.nan
    p = k / n
    return np.sqrt(p * (1.0 - p) / n)


def _clean_axis(ax):
    ax.grid(alpha=0.25, which="both")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _add_panel_label(ax, label):
    ax.text(
        -0.12, 1.08, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left",
    )


def _safe_display(obj):
    try:
        display(obj)
    except Exception:
        print(obj)


def _state_sort_key(x):
    return {s: i for i, s in enumerate(STATE_ORDER)}.get(str(x), 999)


def _switch_class_from_counts(s):
    return pd.cut(
        pd.to_numeric(s, errors="coerce"),
        bins=[-0.5, 0.5, 1.5, 2.5, np.inf],
        labels=["0", "1", "2", "3+"],
    )


# ------------------------------------------------------------
# 1. Standardise interval table and assign current-p states
# ------------------------------------------------------------

def build_state_interval_table_from_hazard_intervals(hazard_intervals):
    d = hazard_intervals.copy()

    team_col = _pick_col(d, ["team"])
    run_col = _pick_col(d, ["run_uid", "run_id_full", "run_id"])
    p_col = _pick_col(d, ["p_group", "p_inst"])
    dt_col = _pick_col(d, ["dt_s"])
    event_col = _pick_col(d, ["event"])
    age_col = _pick_col(d, ["age_mid_s", "age_s", "age_end_s", "age_start_s"])
    dur_col = _pick_col(d, ["run_duration_s", "duration_s"])
    phase_col = _pick_col(d, ["match_phase"], required=False)

    out = pd.DataFrame({
        "team": d[team_col].astype(str),
        "run_uid": d[run_col].astype(str),
        "p": pd.to_numeric(d[p_col], errors="coerce"),
        "dt_s": pd.to_numeric(d[dt_col], errors="coerce"),
        "event": pd.to_numeric(d[event_col], errors="coerce").fillna(0).astype(int),
        "age_s": pd.to_numeric(d[age_col], errors="coerce"),
        "duration_s": pd.to_numeric(d[dur_col], errors="coerce"),
    })

    if phase_col is not None:
        out["match_phase"] = d[phase_col].astype(str)
    else:
        out["match_phase"] = "unknown"

    if "run_length_m" in d.columns:
        out["run_length_m"] = pd.to_numeric(d["run_length_m"], errors="coerce")
    elif "length_m" in d.columns:
        out["run_length_m"] = pd.to_numeric(d["length_m"], errors="coerce")
    else:
        out["run_length_m"] = np.nan

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["team", "run_uid", "p", "dt_s", "event", "age_s", "duration_s"])
    out = out[out["dt_s"] > 0].copy()
    out = out.sort_values(["team", "match_phase", "run_uid", "age_s"]).reset_index(drop=True)

    out["p_state"] = pd.NA
    out["p_q1"] = np.nan
    out["p_q2"] = np.nan

    for team, g in out.groupby("team", observed=True):
        q1, q2 = g["p"].quantile([1 / 3, 2 / 3]).values
        idx = out["team"].eq(team)

        out.loc[idx & (out["p"] <= q1), "p_state"] = "low"
        out.loc[idx & (out["p"] > q1) & (out["p"] <= q2), "p_state"] = "mid"
        out.loc[idx & (out["p"] > q2), "p_state"] = "high"
        out.loc[idx, "p_q1"] = q1
        out.loc[idx, "p_q2"] = q2

    out["p_state"] = pd.Categorical(out["p_state"], categories=STATE_ORDER, ordered=True)

    return out


state_interval_table = build_state_interval_table_from_hazard_intervals(hazard_intervals)


# ------------------------------------------------------------
# 2. Empirical hazards: overall and state-split
# ------------------------------------------------------------

def build_overall_empirical_hazard(d):
    x = d.copy()
    x = x[(x["age_s"] >= AGE_BINS[0]) & (x["age_s"] < AGE_BINS[-1])].copy()

    x["age_bin"] = pd.cut(
        x["age_s"],
        bins=AGE_BINS,
        right=False,
        include_lowest=True,
    )

    rows = []

    for (team, age_bin), g in x.groupby(["team", "age_bin"], observed=True):
        n = len(g)
        events = int(g["event"].sum())

        if n < MIN_EXPOSURE_PER_BIN:
            continue

        q = events / n
        se = _binom_se(events, n)

        rows.append({
            "team": team,
            "age_bin": str(age_bin),
            "age_mid": 0.5 * (float(age_bin.left) + float(age_bin.right)),
            "n_intervals": n,
            "events": events,
            "hazard": q,
            "se": se,
        })

    return pd.DataFrame(rows)


def build_state_empirical_hazard(d):
    x = d.copy()
    x = x[(x["age_s"] >= AGE_BINS[0]) & (x["age_s"] < AGE_BINS[-1])].copy()

    x["age_bin"] = pd.cut(
        x["age_s"],
        bins=AGE_BINS,
        right=False,
        include_lowest=True,
    )

    rows = []

    for (team, state, age_bin), g in x.groupby(["team", "p_state", "age_bin"], observed=True):
        n = len(g)
        events = int(g["event"].sum())

        if n < MIN_EXPOSURE_PER_BIN:
            continue

        q = events / n
        se = _binom_se(events, n)

        rows.append({
            "team": team,
            "p_state": str(state),
            "age_bin": str(age_bin),
            "age_mid": 0.5 * (float(age_bin.left) + float(age_bin.right)),
            "n_intervals": n,
            "events": events,
            "hazard": q,
            "se": se,
        })

    return pd.DataFrame(rows)


overall_hazard_empirical = build_overall_empirical_hazard(state_interval_table)
state_hazard_empirical = build_state_empirical_hazard(state_interval_table)


# ------------------------------------------------------------
# 3. Run-level residence and switch summaries
# ------------------------------------------------------------

def build_run_residence_table(d):
    rows = []

    for (team, run_uid), g in d.groupby(["team", "run_uid"], observed=True):
        exposure = g["dt_s"].sum()
        if exposure <= 0:
            continue

        row = {
            "team": team,
            "run_uid": run_uid,
            "duration_s": g["duration_s"].max(),
            "run_length_m": g["run_length_m"].max(),
            "p_mean": np.average(g["p"], weights=g["dt_s"]),
            "n_intervals": len(g),
            "event": int(g["event"].max()),
            "exposure_s": exposure,
        }

        for state in STATE_ORDER:
            t_state = g.loc[g["p_state"].astype(str) == state, "dt_s"].sum()
            row[f"t_{state}_s"] = t_state
            row[f"f_{state}"] = t_state / exposure

        rows.append(row)

    return pd.DataFrame(rows)


def build_transition_and_switch_tables(d):
    x = d.copy()
    x = x.sort_values(["team", "run_uid", "age_s"]).reset_index(drop=True)

    x["p_state_str"] = x["p_state"].astype(str)
    x["next_state"] = (
        x.groupby(["team", "run_uid"], observed=True)["p_state_str"]
        .shift(-1)
    )

    trans = x.dropna(subset=["next_state"]).copy()
    trans = trans[
        trans["p_state_str"].isin(STATE_ORDER) &
        trans["next_state"].isin(STATE_ORDER)
    ].copy()

    counts = (
        trans.groupby(["team", "p_state_str", "next_state"], observed=True)
        .size()
        .reset_index(name="n")
    )

    probs = counts.copy()
    probs["p"] = (
        probs["n"] /
        probs.groupby(["team", "p_state_str"], observed=True)["n"].transform("sum")
    )

    def count_switches(g):
        s = g["p_state_str"].to_numpy()
        s = s[np.isin(s, STATE_ORDER)]

        if len(s) == 0:
            return pd.Series({
                "n_intervals_state": 0,
                "n_switches": np.nan,
                "start_state": np.nan,
                "end_state": np.nan,
                "dominant_state": np.nan,
            })

        n_switches = int(np.sum(s[1:] != s[:-1])) if len(s) > 1 else 0
        vals, vals_counts = np.unique(s, return_counts=True)

        return pd.Series({
            "n_intervals_state": len(s),
            "n_switches": n_switches,
            "start_state": s[0],
            "end_state": s[-1],
            "dominant_state": vals[np.argmax(vals_counts)],
        })

    run_switch = (
        x.groupby(["team", "run_uid"], observed=True)
        .apply(count_switches)
        .reset_index()
    )

    run_switch["switch_class"] = _switch_class_from_counts(run_switch["n_switches"])

    return counts, probs, run_switch


residence_run_table = build_run_residence_table(state_interval_table)
state_transition_counts, state_transition_probs, run_switch_table = build_transition_and_switch_tables(state_interval_table)

emp_runs_fig4 = residence_run_table.merge(
    run_switch_table[["team", "run_uid", "n_switches", "switch_class", "start_state", "end_state", "dominant_state"]],
    on=["team", "run_uid"],
    how="left",
)

run_switch_summary = (
    emp_runs_fig4
    .groupby(["team", "switch_class"], observed=True)
    .agg(
        n_runs=("run_uid", "size"),
        median_duration_s=("duration_s", "median"),
        mean_duration_s=("duration_s", "mean"),
        median_length_m=("run_length_m", "median"),
        mean_f_high=("f_high", "mean"),
    )
    .reset_index()
)

run_switch_summary["frac_runs"] = (
    run_switch_summary["n_runs"] /
    run_switch_summary.groupby("team", observed=True)["n_runs"].transform("sum")
)


# ------------------------------------------------------------
# 4. Fit state-dependent inverse-age killing model
# ------------------------------------------------------------

def fit_state_killing_model(d_team):
    d = d_team.copy()
    d = d[d["p_state"].astype(str).isin(STATE_ORDER)].copy()

    age = d["age_s"].to_numpy(float)
    dt = d["dt_s"].to_numpy(float)
    event = d["event"].to_numpy(float)
    state = d["p_state"].astype(str).map(STATE_TO_I).to_numpy(int)

    theta0 = np.array([np.log(0.08), np.log(0.4), 0.2, -0.2])

    def unpack(theta):
        lam = np.exp(theta[0])
        mu = np.exp(theta[1])
        beta = np.zeros(3)
        beta[STATE_TO_I["low"]] = theta[2]
        beta[STATE_TO_I["mid"]] = 0.0
        beta[STATE_TO_I["high"]] = theta[3]
        return lam, mu, beta

    def nll(theta):
        lam, mu, beta = unpack(theta)
        h = (lam + mu / (age + A0)) * np.exp(beta[state])
        h = np.clip(h, 1e-12, 50.0)

        hdt = np.clip(h * dt, 1e-12, 50.0)
        q = -np.expm1(-hdt)
        q = np.clip(q, 1e-12, 1.0 - 1e-12)

        ll = event * np.log(q) + (1.0 - event) * (-hdt)
        return -float(np.sum(ll))

    res = minimize(
        nll,
        theta0,
        method="L-BFGS-B",
        bounds=[(-10, 3), (-10, 4), (-5, 5), (-5, 5)],
    )

    lam, mu, beta = unpack(res.x)

    return {
        "lambda": lam,
        "mu": mu,
        "beta": beta,
        "HR_low_vs_mid": np.exp(beta[STATE_TO_I["low"]]),
        "HR_high_vs_mid": np.exp(beta[STATE_TO_I["high"]]),
        "nll": float(res.fun),
        "success": bool(res.success),
        "result": res,
    }


state_killing_fits = {}

for team, g in state_interval_table.groupby("team", observed=True):
    state_killing_fits[team] = fit_state_killing_model(g)


# ------------------------------------------------------------
# 5. Build Markov transition matrices, initial states, and simulate
# ------------------------------------------------------------

transition_mats = {}
initial_state_probs = {}
state_p_values = {}

for team in sorted(state_interval_table["team"].unique()):
    mat = (
        state_transition_probs[state_transition_probs["team"] == team]
        .pivot(index="p_state_str", columns="next_state", values="p")
        .reindex(index=STATE_ORDER, columns=STATE_ORDER)
        .fillna(0.0)
        .to_numpy(float)
    )
    mat = mat / mat.sum(axis=1, keepdims=True)
    transition_mats[team] = mat

    first_rows = (
        state_interval_table[state_interval_table["team"] == team]
        .sort_values(["run_uid", "age_s"])
        .groupby("run_uid", observed=True)
        .first()
        .reset_index()
    )

    probs = (
        first_rows["p_state"].astype(str)
        .value_counts(normalize=True)
        .reindex(STATE_ORDER)
        .fillna(0.0)
        .to_numpy(float)
    )
    probs = probs / probs.sum()
    initial_state_probs[team] = probs

    pvals = (
        state_interval_table[state_interval_table["team"] == team]
        .groupby("p_state", observed=True)["p"]
        .mean()
        .reindex(STATE_ORDER)
        .to_numpy(float)
    )
    state_p_values[team] = pvals


def simulate_markov_killed_runs(team, n_sim=N_SIM_PER_TEAM):
    P = transition_mats[team]
    pi0 = initial_state_probs[team]
    pvals = state_p_values[team]
    fit = state_killing_fits[team]

    lam = fit["lambda"]
    mu = fit["mu"]
    beta = fit["beta"]

    rows = []

    for sim_id in range(n_sim):
        s = rng.choice(3, p=pi0)
        states = []
        t = 0.0

        while t < MAX_T_SIM:
            age = t + 0.5 * DT_SIM
            h = (lam + mu / (age + A0)) * np.exp(beta[s])
            q = 1.0 - np.exp(-h * DT_SIM)

            states.append(s)

            if rng.random() < q:
                break

            s = rng.choice(3, p=P[s])
            t += DT_SIM

        arr = np.asarray(states, dtype=int)
        duration_s = len(arr) * DT_SIM

        rows.append({
            "team": team,
            "sim_id": sim_id,
            "duration_s": duration_s,
            "n_switches": int(np.sum(arr[1:] != arr[:-1])) if len(arr) > 1 else 0,
            "f_high": float(np.mean(arr == STATE_TO_I["high"])),
            "p_mean": float(np.mean(pvals[arr])),
            "start_state": I_TO_STATE[arr[0]],
            "end_state": I_TO_STATE[arr[-1]],
            "censored": duration_s >= MAX_T_SIM,
        })

    return pd.DataFrame(rows)


sim_runs_fig4 = pd.concat(
    [simulate_markov_killed_runs(team) for team in sorted(state_interval_table["team"].unique())],
    ignore_index=True,
)


# ------------------------------------------------------------
# 6. Add run-level p_mean terciles for empirical and simulated runs
# ------------------------------------------------------------

def add_team_tercile_state(df, value_col, out_col):
    d = df.copy()
    d[out_col] = pd.NA

    for team, g in d.groupby("team", observed=True):
        vals = pd.to_numeric(g[value_col], errors="coerce")
        vals = vals[np.isfinite(vals)]

        if len(vals) < 3 or vals.nunique() < 2:
            continue

        q1, q2 = vals.quantile([1 / 3, 2 / 3]).values
        idx = d["team"].eq(team)
        v = pd.to_numeric(d[value_col], errors="coerce")

        d.loc[idx & (v <= q1), out_col] = "low"
        d.loc[idx & (v > q1) & (v <= q2), out_col] = "mid"
        d.loc[idx & (v > q2), out_col] = "high"

    d[out_col] = pd.Categorical(d[out_col], categories=STATE_ORDER, ordered=True)

    return d


emp_runs_fig4 = add_team_tercile_state(emp_runs_fig4, "p_mean", "p_mean_state")
sim_runs_fig4 = add_team_tercile_state(sim_runs_fig4, "p_mean", "p_mean_state")

sim_runs_fig4["switch_class"] = _switch_class_from_counts(sim_runs_fig4["n_switches"])


# ------------------------------------------------------------
# 7. Summary tables
# ------------------------------------------------------------

fig4_model_summary = []

for team in sorted(state_interval_table["team"].unique()):
    fit = state_killing_fits[team]

    fig4_model_summary.append({
        "team": team,
        "lambda": fit["lambda"],
        "mu": fit["mu"],
        "HR_low_vs_mid": fit["HR_low_vs_mid"],
        "HR_high_vs_mid": fit["HR_high_vs_mid"],
        "nll": fit["nll"],
        "success": fit["success"],
        "P_low_to_low": transition_mats[team][STATE_TO_I["low"], STATE_TO_I["low"]],
        "P_mid_to_mid": transition_mats[team][STATE_TO_I["mid"], STATE_TO_I["mid"]],
        "P_high_to_high": transition_mats[team][STATE_TO_I["high"], STATE_TO_I["high"]],
        "pi0_low": initial_state_probs[team][STATE_TO_I["low"]],
        "pi0_mid": initial_state_probs[team][STATE_TO_I["mid"]],
        "pi0_high": initial_state_probs[team][STATE_TO_I["high"]],
    })

fig4_model_summary = pd.DataFrame(fig4_model_summary)

fig4_emp_summary = (
    emp_runs_fig4
    .groupby("team", observed=True)
    .agg(
        n_runs=("run_uid", "size"),
        median_T=("duration_s", "median"),
        mean_T=("duration_s", "mean"),
        p90_T=("duration_s", lambda x: np.quantile(x, 0.90)),
        mean_switches=("n_switches", "mean"),
        mean_f_high=("f_high", "mean"),
    )
    .reset_index()
)

fig4_sim_summary = (
    sim_runs_fig4
    .groupby("team", observed=True)
    .agg(
        n_sim=("sim_id", "size"),
        median_T=("duration_s", "median"),
        mean_T=("duration_s", "mean"),
        p90_T=("duration_s", lambda x: np.quantile(x, 0.90)),
        mean_switches=("n_switches", "mean"),
        mean_f_high=("f_high", "mean"),
        frac_censored=("censored", "mean"),
    )
    .reset_index()
)

print("=" * 100)
print("Figure 4 model summary")
print("=" * 100)
_safe_display(fig4_model_summary)

print("=" * 100)
print("Figure 4 empirical summary")
print("=" * 100)
_safe_display(fig4_emp_summary)

print("=" * 100)
print("Figure 4 simulation summary")
print("=" * 100)
_safe_display(fig4_sim_summary)

print("=" * 100)
print("Switch-count summary")
print("=" * 100)
_safe_display(run_switch_summary)


# ------------------------------------------------------------
# 8. Figure panel plotting
# ------------------------------------------------------------

def plot_figure4_for_team(team):
    e = emp_runs_fig4[emp_runs_fig4["team"] == team].copy()
    s = sim_runs_fig4[sim_runs_fig4["team"] == team].copy()

    hz_all = overall_hazard_empirical[overall_hazard_empirical["team"] == team].copy()
    hz_state = state_hazard_empirical[state_hazard_empirical["team"] == team].copy()

    mat = transition_mats[team]
    fit = state_killing_fits[team]

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15.8, 9.2),
        constrained_layout=True,
    )

    fig.suptitle(
        f"Figure 4 draft — age memory and collective-state switching | {team}",
        fontsize=15,
        fontweight="bold",
        y=1.035,
    )

    # --------------------------------------------------------
    # A. Overall hazard
    # --------------------------------------------------------
    ax = axes[0, 0]
    _add_panel_label(ax, "A")

    ax.errorbar(
        hz_all["age_mid"],
        hz_all["hazard"],
        yerr=1.96 * hz_all["se"],
        fmt="o",
        color="k",
        capsize=3,
        label="empirical",
    )

    a_grid = np.linspace(1, 60, 300)
    h0 = fit["lambda"] + fit["mu"] / (a_grid + A0)
    q0 = 1.0 - np.exp(-h0 * DT_SIM)

    ax.plot(
        a_grid,
        q0,
        lw=2,
        color="tab:blue",
        label=r"inverse-age fit",
    )

    ax.set_yscale("log")
    ax.set_xlabel("run age [s]")
    ax.set_ylabel("termination probability per interval")
    ax.set_title("overall age-dependent hazard")
    ax.legend(frameon=False)
    _clean_axis(ax)

    # --------------------------------------------------------
    # B. Hazard by current state
    # --------------------------------------------------------
    ax = axes[0, 1]
    _add_panel_label(ax, "B")

    for state in STATE_ORDER:
        sub = hz_state[hz_state["p_state"] == state].copy()
        if len(sub) == 0:
            continue

        ax.errorbar(
            sub["age_mid"],
            sub["hazard"],
            yerr=1.96 * sub["se"],
            fmt="o",
            capsize=3,
            color=STATE_COLORS[state],
            label=f"emp {state}",
        )

        beta_s = fit["beta"][STATE_TO_I[state]]
        q_state = 1.0 - np.exp(-h0 * np.exp(beta_s) * DT_SIM)

        ax.plot(
            a_grid,
            q_state,
            lw=2,
            color=STATE_COLORS[state],
            alpha=0.9,
            label=f"fit {state}",
        )

    ax.set_yscale("log")
    ax.set_xlabel("run age [s]")
    ax.set_ylabel("termination probability per interval")
    ax.set_title("state-dependent killing")
    ax.legend(frameon=False, fontsize=6)
    _clean_axis(ax)

    # --------------------------------------------------------
    # C. Transition matrix
    # --------------------------------------------------------
    ax = axes[0, 2]
    _add_panel_label(ax, "C")

    im = ax.imshow(mat, vmin=0, vmax=1)

    ax.set_xticks(np.arange(len(STATE_ORDER)))
    ax.set_yticks(np.arange(len(STATE_ORDER)))
    ax.set_xticklabels(STATE_ORDER)
    ax.set_yticklabels(STATE_ORDER)
    ax.set_xlabel("next state")
    ax.set_ylabel("current state")
    ax.set_title(r"state persistence: $P(s_{t+1}|s_t)$")

    for i in range(len(STATE_ORDER)):
        for j in range(len(STATE_ORDER)):
            val = mat[i, j]
            ax.text(
                j, i, f"{val:.2f}",
                ha="center",
                va="center",
                color="white" if val > 0.5 else "black",
                fontsize=9,
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # --------------------------------------------------------
    # D. Duration tails by switch count
    # --------------------------------------------------------
    ax = axes[1, 0]
    _add_panel_label(ax, "D")

    for cls in ["0", "1", "2", "3+"]:
        sub = e[e["switch_class"].astype(str) == cls]
        x, y = _ccdf(sub["duration_s"])

        if len(x) == 0:
            continue

        ax.step(
            x, y,
            where="post",
            lw=2,
            color=SWITCH_COLORS[cls],
            label=f"{cls} switches (n={len(sub)})",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("centroid-run duration T [s]")
    ax.set_ylabel(r"$P(T \geq t)$")
    ax.set_title("duration tails by state-switch count")
    ax.legend(frameon=False, fontsize=6)
    _clean_axis(ax)

    # --------------------------------------------------------
    # E. Total CCDF reconstruction
    # --------------------------------------------------------
    ax = axes[1, 1]
    _add_panel_label(ax, "E")

    x_emp, y_emp = _ccdf(e["duration_s"])
    x_sim, y_sim = _ccdf(s["duration_s"])

    ax.step(
        x_emp, y_emp,
        where="post",
        color="k",
        lw=2.5,
        label="empirical",
    )

    ax.step(
        x_sim, y_sim,
        where="post",
        color="tab:orange",
        lw=2.5,
        ls="--",
        label="Markov-killed model",
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("centroid-run duration T [s]")
    ax.set_ylabel(r"$P(T \geq t)$")
    ax.set_title("total duration reconstruction")
    ax.legend(frameon=False)
    _clean_axis(ax)

    # --------------------------------------------------------
    # F. p_mean CCDF reconstruction
    # --------------------------------------------------------
    ax = axes[1, 2]
    _add_panel_label(ax, "F")

    for state in STATE_ORDER:
        emp_sub = e[e["p_mean_state"].astype(str) == state]
        sim_sub = s[s["p_mean_state"].astype(str) == state]

        x_emp, y_emp = _ccdf(emp_sub["duration_s"])
        x_sim, y_sim = _ccdf(sim_sub["duration_s"])

        if len(x_emp):
            ax.step(
                x_emp, y_emp,
                where="post",
                lw=2,
                color=STATE_COLORS[state],
                label=f"emp {state}",
            )

        if len(x_sim):
            ax.step(
                x_sim, y_sim,
                where="post",
                lw=2,
                ls="--",
                color=STATE_COLORS[state],
                alpha=0.8,
                label=f"model {state}",
            )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("centroid-run duration T [s]")
    ax.set_ylabel(r"$P(T \geq t)$")
    ax.set_title(r"reconstruction by realised $p_{\rm mean}$")
    ax.legend(frameon=False, fontsize=6)
    _clean_axis(ax)

    if SAVE_FIGURES:
        _ensure_outdir(FIG_OUTDIR)
        safe_team = str(team).replace(" ", "_").replace("/", "_")
        outpath = f"{FIG_OUTDIR}/{FIG_BASENAME}_{safe_team}.png"
        fig.savefig(outpath, dpi=FIG_DPI, bbox_inches="tight")
        print("Saved:", outpath)

    plt.show()

    return fig


fig4_panel_by_team = {}

for team in sorted(emp_runs_fig4["team"].unique()):
    fig4_panel_by_team[team] = plot_figure4_for_team(team)


print("Created objects:")
print("  state_interval_table")
print("  overall_hazard_empirical")
print("  state_hazard_empirical")
print("  residence_run_table")
print("  state_transition_counts")
print("  state_transition_probs")
print("  run_switch_table")
print("  run_switch_summary")
print("  state_killing_fits")
print("  transition_mats")
print("  initial_state_probs")
print("  state_p_values")
print("  emp_runs_fig4")
print("  sim_runs_fig4")
print("  fig4_model_summary")
print("  fig4_emp_summary")
print("  fig4_sim_summary")
print("  fig4_panel_by_team")

### Fig. 5. Expand model from discrete markov into continuous stochastic model.

In [ ]:
# ============================================================
# CONTINUOUS STOCHASTIC p(t) KILLED PROCESS v1
# ------------------------------------------------------------
# Tests continuous theory:
#
#   dp = b(p) dt + sigma(p) dW
#
#   h(a,p) = [lambda + mu/(a+a0)] * exp(beta * z(p))
#
# against:
#   1) total duration CCDF
#   2) p_mean-split duration CCDF
#
# Requires existing:
#   state_interval_table
#   emp_runs_fig4
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# -----------------------------
# Settings
# -----------------------------
DT = 1.0
A0 = 1.0
MAX_T = 120.0
N_SIM_PER_TEAM = 20000
SEED = 19

N_P_BINS = 12
MIN_TRANS_PER_BIN = 20

STATE_ORDER = ["low", "mid", "high"]

rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 7,
})


# -----------------------------
# Helpers
# -----------------------------
def ccdf(values):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return np.array([]), np.array([])
    x = np.sort(x)
    y = 1.0 - np.arange(len(x)) / len(x)
    return x, y


def clean_axis(ax):
    ax.grid(alpha=0.25, which="both")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def reflect_unit_interval(x):
    """
    Reflect values into [0, 1].
    Works for small overshoots.
    """
    x = np.asarray(x)
    x = np.where(x < 0, -x, x)
    x = np.where(x > 1, 2 - x, x)
    return np.clip(x, 0, 1)


def add_team_tercile_state(df, value_col, out_col):
    d = df.copy()
    d[out_col] = pd.NA

    for team, g in d.groupby("team", observed=True):
        vals = pd.to_numeric(g[value_col], errors="coerce")
        vals = vals[np.isfinite(vals)]

        if len(vals) < 3 or vals.nunique() < 2:
            continue

        q1, q2 = vals.quantile([1 / 3, 2 / 3]).values
        idx = d["team"].eq(team)
        v = pd.to_numeric(d[value_col], errors="coerce")

        d.loc[idx & (v <= q1), out_col] = "low"
        d.loc[idx & (v > q1) & (v <= q2), out_col] = "mid"
        d.loc[idx & (v > q2), out_col] = "high"

    d[out_col] = pd.Categorical(d[out_col], categories=STATE_ORDER, ordered=True)
    return d


# ============================================================
# 1. Estimate empirical continuous p dynamics
# ============================================================

def estimate_p_sde_from_intervals(d_team, n_bins=N_P_BINS):
    """
    Estimate:
      E[dp | p] / dt
      Var[dp | p] / dt

    using observed consecutive p transitions within runs.
    """
    d = d_team.copy()
    d = d.sort_values(["run_uid", "age_s"]).reset_index(drop=True)

    d["p_next"] = d.groupby("run_uid", observed=True)["p"].shift(-1)
    d["dt_next"] = d.groupby("run_uid", observed=True)["age_s"].shift(-1) - d["age_s"]

    trans = d.dropna(subset=["p", "p_next", "dt_next"]).copy()
    trans = trans[(trans["dt_next"] > 0) & np.isfinite(trans["dt_next"])].copy()

    trans["dp"] = trans["p_next"] - trans["p"]
    trans["dp_per_dt"] = trans["dp"] / trans["dt_next"]
    trans["var_per_dt"] = (trans["dp"] ** 2) / trans["dt_next"]

    bins = np.linspace(0, 1, n_bins + 1)
    mids = 0.5 * (bins[:-1] + bins[1:])

    trans["p_bin"] = pd.cut(
        trans["p"],
        bins=bins,
        include_lowest=True,
        right=False,
        labels=False,
    )

    rows = []

    global_b = trans["dp_per_dt"].mean()
    global_sig2 = trans["var_per_dt"].mean()

    for k in range(n_bins):
        g = trans[trans["p_bin"] == k]

        if len(g) >= MIN_TRANS_PER_BIN:
            b = g["dp_per_dt"].mean()
            sig2 = g["var_per_dt"].mean()
            n = len(g)
        else:
            b = global_b
            sig2 = global_sig2
            n = len(g)

        rows.append({
            "p_mid": mids[k],
            "p_left": bins[k],
            "p_right": bins[k + 1],
            "n": n,
            "b": float(b),
            "sigma": float(np.sqrt(max(sig2, 1e-6))),
        })

    dyn = pd.DataFrame(rows)

    # Smooth a little by rolling average to avoid jagged drift/diffusion
    dyn["b_smooth"] = dyn["b"].rolling(3, center=True, min_periods=1).mean()
    dyn["sigma_smooth"] = dyn["sigma"].rolling(3, center=True, min_periods=1).mean()

    return dyn, trans


p_dynamics_by_team = {}
p_transition_samples_by_team = {}

for team, g in state_interval_table.groupby("team", observed=True):
    dyn, trans = estimate_p_sde_from_intervals(g)
    p_dynamics_by_team[team] = dyn
    p_transition_samples_by_team[team] = trans

    print("\n" + "=" * 100)
    print(f"{team}: empirical p dynamics")
    print("=" * 100)
    display(dyn)


# ============================================================
# 2. Fit continuous hazard h(a,p)
# ============================================================

def fit_continuous_p_hazard(d_team):
    """
    Fits:
      h(a,p) = [lambda + mu/(a+a0)] * exp(beta * z(p))
    """
    d = d_team.copy()
    d = d.dropna(subset=["age_s", "dt_s", "event", "p"]).copy()

    age = d["age_s"].to_numpy(float)
    dt = d["dt_s"].to_numpy(float)
    event = d["event"].to_numpy(float)
    p = d["p"].to_numpy(float)

    p_mean = np.mean(p)
    p_sd = np.std(p)
    if p_sd <= 1e-12:
        p_sd = 1.0
    z = (p - p_mean) / p_sd

    # log_lambda, log_mu, beta
    theta0 = np.array([np.log(0.08), np.log(0.4), -0.15])

    def nll(theta):
        lam = np.exp(theta[0])
        mu = np.exp(theta[1])
        beta = theta[2]

        h0 = lam + mu / (age + A0)
        h = h0 * np.exp(beta * z)
        h = np.clip(h, 1e-12, 50)

        hdt = np.clip(h * dt, 1e-12, 50)
        q = -np.expm1(-hdt)
        q = np.clip(q, 1e-12, 1 - 1e-12)

        ll = event * np.log(q) + (1 - event) * (-hdt)
        return -float(np.sum(ll))

    res = minimize(
        nll,
        theta0,
        method="L-BFGS-B",
        bounds=[(-10, 3), (-10, 4), (-5, 5)],
    )

    fit = {
        "lambda": float(np.exp(res.x[0])),
        "mu": float(np.exp(res.x[1])),
        "beta": float(res.x[2]),
        "HR_1sd_p": float(np.exp(res.x[2])),
        "p_mean": float(p_mean),
        "p_sd": float(p_sd),
        "nll": float(res.fun),
        "success": bool(res.success),
        "result": res,
    }

    return fit


continuous_hazard_fits = {}

for team, g in state_interval_table.groupby("team", observed=True):
    fit = fit_continuous_p_hazard(g)
    continuous_hazard_fits[team] = fit

    print("\n" + "=" * 100)
    print(f"{team}: continuous p hazard fit")
    print("=" * 100)
    print(
        f"success={fit['success']} | "
        f"lambda={fit['lambda']:.4f}, mu={fit['mu']:.4f}, "
        f"beta={fit['beta']:+.3f}, HR_1sd_p={fit['HR_1sd_p']:.3f}, "
        f"NLL={fit['nll']:.1f}"
    )


# ============================================================
# 3. Initial p distribution
# ============================================================

initial_p_samples_by_team = {}

for team, g in state_interval_table.groupby("team", observed=True):
    first = (
        g.sort_values(["run_uid", "age_s"])
         .groupby("run_uid", observed=True)
         .first()
         .reset_index()
    )
    initial_p_samples_by_team[team] = first["p"].dropna().to_numpy(float)

    print(
        team,
        "n initial p samples:",
        len(initial_p_samples_by_team[team]),
        "mean:",
        np.mean(initial_p_samples_by_team[team]),
    )


# ============================================================
# 4. Simulate continuous p(t) killed process
# ============================================================

def interp_drift_sigma(team, p):
    dyn = p_dynamics_by_team[team]
    mids = dyn["p_mid"].to_numpy(float)
    b = dyn["b_smooth"].to_numpy(float)
    sig = dyn["sigma_smooth"].to_numpy(float)

    bp = np.interp(p, mids, b, left=b[0], right=b[-1])
    sp = np.interp(p, mids, sig, left=sig[0], right=sig[-1])
    return bp, max(sp, 1e-4)


def simulate_continuous_team(team, n_sim=N_SIM_PER_TEAM):
    fit = continuous_hazard_fits[team]
    p0_samples = initial_p_samples_by_team[team]

    rows = []

    for sim_id in range(n_sim):
        p = float(rng.choice(p0_samples))
        t = 0.0

        p_path = []
        alive = True

        while alive and t < MAX_T:
            age = t + 0.5 * DT

            z = (p - fit["p_mean"]) / fit["p_sd"]
            h0 = fit["lambda"] + fit["mu"] / (age + A0)
            h = h0 * np.exp(fit["beta"] * z)
            q = 1.0 - np.exp(-h * DT)

            p_path.append(p)

            if rng.random() < q:
                break

            b, sig = interp_drift_sigma(team, p)
            p = p + b * DT + sig * np.sqrt(DT) * rng.normal()
            p = float(reflect_unit_interval(p))

            t += DT

        arr = np.asarray(p_path, dtype=float)
        duration_s = len(arr) * DT

        rows.append({
            "team": team,
            "sim_id": sim_id,
            "duration_s": duration_s,
            "p_mean": float(np.mean(arr)),
            "p_start": float(arr[0]),
            "p_end": float(arr[-1]),
            "p_sd_path": float(np.std(arr)),
            "censored": duration_s >= MAX_T,
        })

    return pd.DataFrame(rows)


sim_continuous_p = pd.concat(
    [
        simulate_continuous_team(team)
        for team in sorted(state_interval_table["team"].unique())
    ],
    ignore_index=True,
)

sim_continuous_p = add_team_tercile_state(sim_continuous_p, "p_mean", "p_mean_state")

emp_runs_continuous_ref = emp_runs_fig4.copy()
emp_runs_continuous_ref = add_team_tercile_state(emp_runs_continuous_ref, "p_mean", "p_mean_state")


# ============================================================
# 5. Summary tables
# ============================================================

continuous_summary = (
    sim_continuous_p
    .groupby("team", observed=True)
    .agg(
        n_sim=("sim_id", "size"),
        median_T=("duration_s", "median"),
        mean_T=("duration_s", "mean"),
        p90_T=("duration_s", lambda x: np.quantile(x, 0.90)),
        mean_p_mean=("p_mean", "mean"),
        mean_path_sd=("p_sd_path", "mean"),
        frac_censored=("censored", "mean"),
    )
    .reset_index()
)

emp_continuous_ref_summary = (
    emp_runs_continuous_ref
    .groupby("team", observed=True)
    .agg(
        n_runs=("run_uid", "size"),
        median_T=("duration_s", "median"),
        mean_T=("duration_s", "mean"),
        p90_T=("duration_s", lambda x: np.quantile(x, 0.90)),
        mean_p_mean=("p_mean", "mean"),
    )
    .reset_index()
)

print("=" * 100)
print("Empirical reference summary")
print("=" * 100)
display(emp_continuous_ref_summary)

print("=" * 100)
print("Continuous stochastic simulation summary")
print("=" * 100)
display(continuous_summary)


# ============================================================
# 6. Plots
# ============================================================

for team in sorted(emp_runs_continuous_ref["team"].unique()):
    emp = emp_runs_continuous_ref[emp_runs_continuous_ref["team"] == team].copy()
    sim = sim_continuous_p[sim_continuous_p["team"] == team].copy()
    dyn = p_dynamics_by_team[team]
    trans = p_transition_samples_by_team[team]

    fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)

    fig.suptitle(
        f"Continuous stochastic order process + killing | {team}",
        fontsize=14,
        fontweight="bold",
        y=1.03,
    )

    # A. empirical p dynamics
    ax = axes[0, 0]
    ax.axhline(0, color="k", lw=1, alpha=0.5)
    ax.plot(dyn["p_mid"], dyn["b_smooth"], "o-", label=r"$b(p)$")
    ax.set_xlabel("polarisation p")
    ax.set_ylabel("drift")
    ax.set_title("estimated drift")
    ax.legend(frameon=False)
    clean_axis(ax)

    # B. diffusion
    ax = axes[0, 1]
    ax.plot(dyn["p_mid"], dyn["sigma_smooth"], "o-", label=r"$\sigma(p)$")
    ax.set_xlabel("polarisation p")
    ax.set_ylabel("diffusion scale")
    ax.set_title("estimated diffusion")
    ax.legend(frameon=False)
    clean_axis(ax)

    # C. hazard multiplier
    ax = axes[0, 2]
    fit = continuous_hazard_fits[team]
    p_grid = np.linspace(0.01, 0.99, 200)
    z_grid = (p_grid - fit["p_mean"]) / fit["p_sd"]
    mult = np.exp(fit["beta"] * z_grid)
    ax.plot(p_grid, mult, lw=2)
    ax.axhline(1, color="k", lw=1, alpha=0.5)
    ax.set_xlabel("polarisation p")
    ax.set_ylabel("hazard multiplier")
    ax.set_title(r"$\phi(p)=\exp[\beta z(p)]$")
    clean_axis(ax)

    # D. total CCDF
    ax = axes[1, 0]
    x, y = ccdf(emp["duration_s"])
    ax.step(x, y, where="post", lw=2.5, color="k", label="empirical")
    x, y = ccdf(sim["duration_s"])
    ax.step(x, y, where="post", lw=2.5, ls="--", label="continuous model")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("duration T [s]")
    ax.set_ylabel(r"$P(T \geq t)$")
    ax.set_title("total duration CCDF")
    ax.legend(frameon=False)
    clean_axis(ax)

    # E. p_mean split
    ax = axes[1, 1]
    for state in STATE_ORDER:
        emp_sub = emp[emp["p_mean_state"].astype(str) == state]
        sim_sub = sim[sim["p_mean_state"].astype(str) == state]

        x, y = ccdf(emp_sub["duration_s"])
        if len(x):
            ax.step(x, y, where="post", lw=2, label=f"emp {state}")

        x, y = ccdf(sim_sub["duration_s"])
        if len(x):
            ax.step(x, y, where="post", lw=2, ls="--", label=f"sim {state}")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(1e-3, 1.05)
    ax.set_xlabel("duration T [s]")
    ax.set_ylabel(r"$P(T \geq t)$")
    ax.set_title(r"duration CCDF by realised $p_{\rm mean}$")
    ax.legend(frameon=False, fontsize=7)
    clean_axis(ax)

    # F. p_mean distribution
    ax = axes[1, 2]
    bins = np.linspace(0, 1, 31)
    ax.hist(emp["p_mean"], bins=bins, density=True, alpha=0.45, label="empirical")
    ax.hist(sim["p_mean"], bins=bins, density=True, alpha=0.45, label="continuous model")
    ax.set_xlabel(r"realised $p_{\rm mean}$")
    ax.set_ylabel("density")
    ax.set_title(r"distribution of realised $p_{\rm mean}$")
    ax.legend(frameon=False)
    clean_axis(ax)

    plt.show()


print("Created objects:")
print("  p_dynamics_by_team")
print("  p_transition_samples_by_team")
print("  continuous_hazard_fits")
print("  initial_p_samples_by_team")
print("  sim_continuous_p")
print("  continuous_summary")
print("  emp_continuous_ref_summary")

### Animation

In [ ]:
import importlib
import viz.animation as anim
importlib.reload(anim)

from viz.animation import animate_players_media
from IPython.display import HTML

ani_media = animate_players_media(
    df_labeled, pitch_xy,
    matches=A_matches, game_number=gnum+1,
    step_s=1, max_frames=1000,
    show_trails=True, trail_only_active=True, trail_s=10, trail_alpha=0.6,

    show_arrows=True,
    arrow_len_m=5.0,
    arrow_width=0.0065,
    arrow_window_s=2.0,

    df_pmv=df_pmv,
    show_pmv_bar=True,
    show_team_speed_bar=True,
    team_speed_value_col="v_group_mps",  # default now, but explicit is fine
    pmv_pad_limit_s=3.0,                 # helps if timestamps aren’t exact
)

HTML(ani_media.to_jshtml())